# 02 — Layer 1 Alpha Signals

This notebook turns the output of `01_data_hub.ipynb` into a reusable weekly alpha-signal library for an ETF research stack.

**What this notebook does**

- audits the upstream data contract before computing anything
- defines an explicit timing convention and enforces a signal-lag policy
- double-checks that every definition is consistent with weekly bars
- builds price, proxy-fundamental, and macro-conditioning signals
- validates them with cross-sectional IC, t-stats, decay, coverage, turnover, and simple cost sensitivity
- saves signal files under `data/02_layer1_signals/` for downstream notebooks

**Design choices**

- The notebook expects `01_data_hub.ipynb` to save reusable inputs into `data/01_data_hub/`.
- If those files are missing, it can rebuild a minimal upstream dataset with the same broad conventions.
- Layer 1 writes only to `data/02_layer1_signals/`, so the handoff between notebooks stays clean.
- External features are treated more conservatively than price features to reduce lookahead bias.

**Deliverables created here**

- `signal_tsmom.csv`
- `signal_xsmom.csv`
- `signal_reversal.csv`
- `signal_multi_horizon_mom.csv`
- `signal_residual_momentum.csv`
- `signal_carry.csv`
- `signal_value.csv`
- `signal_bab.csv`
- `signal_quality.csv`
- `regime_features.csv`
- `signal_summary_table.csv`
- `signal_ic_by_horizon.csv`
- `signal_redundancy_matrix.csv`
- `signal_eligibility_matrix.csv`
- `signal_manifest.json`


In [ ]:
from __future__ import annotations

import ast
import json
import time
import warnings
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except ImportError:
    yf = None

try:
    from fredapi import Fred
except ImportError:
    Fred = None

try:
    from pytrends.request import TrendReq
except ImportError:
    TrendReq = None

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.4f}".format
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
np.random.seed(7)


## 1. Global Timing Convention, Data Contract, and Weekly Definitions

This section is intentionally explicit because most signal bugs start with frequency mismatches or subtle timing mistakes.

### What we mean by “weekly” here

- `01_data_hub.ipynb` resamples prices to **Friday closes** using `resample("W-FRI").last()`.
- It saves **weekly log returns**, not simple returns.
- Several alpha formulas, however, are more naturally defined with **price ratios / simple returns**.

### Important correction for this notebook

The requested 12-1 style weekly momentum formula is:

\[
\text{raw\_tsmom}_{t} = \frac{P_{t-4}}{P_{t-52}} - 1
\]

That is the right weekly analogue of “skip the most recent month” when the data are weekly.
We therefore:

- use **prices / simple returns** for horizon-return formulas such as 52-to-4 week momentum
- use **weekly log returns** from the data hub for realized volatility estimates, because that is the upstream convention and it annualizes cleanly

### Global timing convention

This notebook now uses a single timing rule that downstream notebooks should inherit.

- **Signal formation date**: the Friday close labeled `t`
- **Rebalance date**: the next tradable weekly rebalance point after the signal is known
- **Forward return measurement**: returns from `t+1` onward, measured **after** the lagged signal date
- **Price-based features**: can use information visible through the Friday close, but are still shifted by `SIGNAL_LAG_WEEKS`
- **Non-price external features**: macro data, Google Trends, Yahoo metadata, and yield / distribution metadata receive an additional conservative lag because publication timing is not guaranteed to line up with the Friday close

In practice, this means:

- price-based signals are validated on a **1-week lag**
- external non-price features are validated or exported on a **2-week effective lag** by default:
  one week for feature availability conservatism and one week for trade execution realism

### Why this prevents lookahead bias

Without a lag, the notebook could accidentally reward itself for using data that were not fully observable when the portfolio would have been rebalanced.
That is especially dangerous for:

- macro releases with publication delays
- Google Trends data that can be revised or retrieved after the fact
- Yahoo metadata and trailing-yield fields that are point-in-time snapshots, not guaranteed historical truth

This is more important than squeezing out slightly better backtest numbers.
A weaker but realistically timed signal is more useful than a stronger signal built on accidental leakage.

### Why this matters downstream

Layer 2 signal combination, Layer 3 portfolio construction, and any backtest or walk-forward notebook should inherit the same convention.
If they do not, later performance can look better simply because signal timing got looser, not because the research got stronger.


In [ ]:
PROJECT_DIR = Path.cwd()
DATA_ROOT = PROJECT_DIR / "data"
DATA_HUB_DIR = DATA_ROOT / "01_data_hub"
OUTPUT_DIR = DATA_ROOT / "02_layer1_signals"

DATA_HUB_PATH = PROJECT_DIR / "01_data_hub.ipynb"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_HUB_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIR_CANDIDATES = [
    DATA_HUB_DIR,
    PROJECT_DIR / "data" / "processed",
    PROJECT_DIR.parent / "data" / "processed",
    PROJECT_DIR.parent / "data2" / "processed",
]

REQUIRED_INPUTS = [
    "weekly_prices.csv",
    "weekly_returns.csv",
    "macro_weekly.csv",
    "vix_term_structure.csv",
    "google_trends.csv",
    "data_quality_report.csv",
    "universe.json",
]

OPTIONAL_INPUTS = [
    "daily_returns.csv",
    "benchmark_prices_weekly.csv",
    "benchmark_returns_weekly.csv",
    "market_proxy_weekly.csv",
    "benchmarks.csv",
    "proxy_mapping.json",
    "universe_metadata.csv",
    "yahoo_metadata_snapshot.csv",
    "etf_distribution_history.csv",
    "google_trends_raw.csv",
    "google_trends_snapshot_meta.json",
]

WEEKS_PER_YEAR = 52
FORWARD_HORIZONS = (1, 2, 4, 8, 12)
TSMOM_LOOKBACK = 52
TSMOM_SKIP = 4
TSMOM_FORMATION_WEEKS = TSMOM_LOOKBACK - TSMOM_SKIP
TSMOM_VOL_WINDOW = 26
MULTI_BLEND_VOL_WINDOW = 52
MULTI_BLEND_MIN_COMPONENT_VOL = 0.01
RESIDUAL_BETA_WINDOW = 52
RESIDUAL_MOM_FORMATION_WEEKS = TSMOM_FORMATION_WEEKS
BETA_WINDOW = 52
BETA_MIN_PERIODS = 26
QUALITY_WINDOW = 26
QUALITY_DRAWDOWN_WINDOW = 52
QUALITY_MIN_PERIODS = 13
VALUE_WINDOW = 260
VALUE_MIN_PERIODS = 104
MACRO_Z_WINDOW = 104
MACRO_MIN_PERIODS = 52
CARRY_TRAILING_WEEKS = 52
MIN_CROSS_SECTION = 5
ASSET_CLASS_NEUTRAL_MIN_GROUP_SIZE = 2
COST_BPS_GRID = (0, 5, 10, 25)
WINSOR_LOWER_Q = 0.05
WINSOR_UPPER_Q = 0.95
CROSS_SECTIONAL_ZCAP = 3.0
SIGNAL_LAG_WEEKS = 1
EXTERNAL_DATA_LAG_WEEKS = 1
USE_NEWEY_WEST_TSTATS = True
NEWEY_WEST_MAX_LAGS = None
AUTO_REBUILD_MISSING_INPUTS = True

print("Project directory:", PROJECT_DIR.resolve())
print("Data hub input directory:", DATA_HUB_DIR.resolve())
print("Layer 1 signal output directory:", OUTPUT_DIR.resolve())
print()
print("Timing settings")
print(f"  SIGNAL_LAG_WEEKS        = {SIGNAL_LAG_WEEKS}")
print(f"  EXTERNAL_DATA_LAG_WEEKS = {EXTERNAL_DATA_LAG_WEEKS}")
print()
print("Candidate upstream input locations:")
for path in INPUT_DIR_CANDIDATES:
    print("  -", path.resolve())


In [ ]:
def ensure_series(x, name=None):
    """Force a 1D numeric time series with a timezone-naive DatetimeIndex."""
    if isinstance(x, pd.DataFrame):
        x = x.squeeze()
    x = pd.to_numeric(x, errors="coerce")
    x.index = pd.to_datetime(x.index).tz_localize(None)
    if name is not None:
        x.name = name
    return x


def flatten_columns(df):
    """Flatten multi-indexed columns so CSV round-tripping stays simple."""
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        new_cols = []
        for col in df.columns:
            if isinstance(col, tuple):
                vals = [str(v) for v in col if v not in [None, "", "Close", "Adj Close"]]
                new_cols.append(vals[-1] if vals else str(col[-1]))
            else:
                new_cols.append(str(col))
        df.columns = new_cols
    else:
        df.columns = [str(c) for c in df.columns]
    df.columns.name = None
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "Date"
    return df


def safe_download_close(ticker, start, end=None, auto_adjust=True):
    """Download one ticker and return a clean close-price Series."""
    if yf is None:
        raise ImportError("yfinance is required for the graceful rebuild path.")
    data = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=auto_adjust,
        progress=False,
        threads=False,
    )
    if data.empty:
        return None
    if "Close" in data.columns:
        close = data["Close"]
    elif "Adj Close" in data.columns:
        close = data["Adj Close"]
    else:
        close = data.squeeze()
    return ensure_series(close, name=ticker)


def load_notebook_json(path):
    return json.loads(Path(path).read_text())


def extract_literal_assignment(notebook_path, variable_name):
    if not Path(notebook_path).exists():
        return None
    notebook = load_notebook_json(notebook_path)
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source_text = "".join(cell.get("source", []))
        try:
            tree = ast.parse(source_text)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and target.id == variable_name:
                        try:
                            return ast.literal_eval(node.value)
                        except Exception:
                            return None
    return None


def choose_input_dir(candidates, required_files):
    best_dir = None
    best_count = -1
    best_missing = list(required_files)
    for directory in candidates:
        existing = [f for f in required_files if (directory / f).exists()]
        if len(existing) > best_count:
            best_dir = directory
            best_count = len(existing)
            best_missing = [f for f in required_files if f not in existing]
    return best_dir, best_missing


def read_panel_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    frame.index.name = "Date"
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    return flatten_columns(frame)


def read_table_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json_file(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def get_proxy_ticker(proxy_mapping, role, fallback=None):
    if not isinstance(proxy_mapping, dict):
        return fallback
    entry = proxy_mapping.get(role, fallback)
    if isinstance(entry, dict):
        return entry.get("ticker", fallback)
    if isinstance(entry, str):
        return entry
    return fallback


def infer_asset_class_from_description(description):
    description = str(description or "").lower()
    if "reit" in description or "real estate" in description:
        return "REITs"
    if "treasur" in description or "bond" in description or "t-bill" in description or "mortgage" in description or "tips" in description:
        return "Bonds"
    if "gold" in description or "silver" in description or "oil" in description or "commodit" in description or "agriculture" in description:
        return "Commodities"
    if "dollar" in description or "fx" in description or "currency" in description:
        return "FX"
    return "Equities"


def build_asset_class_map(universe_def, universe_metadata, columns):
    columns = list(columns)
    asset_class_map = {}

    if isinstance(universe_def, dict) and isinstance(universe_def.get("asset_class_map"), dict):
        asset_class_map.update(universe_def["asset_class_map"])

    if not universe_metadata.empty:
        meta = universe_metadata.copy()
        meta.columns = [str(c).strip().lower() for c in meta.columns]
        if {"ticker", "asset_class"}.issubset(meta.columns):
            asset_class_map.update(meta.set_index("ticker")["asset_class"].to_dict())

    descriptions = {}
    if isinstance(universe_def, dict) and isinstance(universe_def.get("descriptions"), dict):
        descriptions = universe_def["descriptions"]

    final_map = {}
    for ticker in columns:
        if ticker in asset_class_map and pd.notna(asset_class_map[ticker]):
            final_map[ticker] = asset_class_map[ticker]
        else:
            final_map[ticker] = infer_asset_class_from_description(descriptions.get(ticker, ticker))
    return final_map


def trailing_simple_return(prices, lookback, skip=0):
    numerator = prices.shift(skip)
    denominator = prices.shift(lookback)
    return numerator.div(denominator) - 1.0


def annualized_realized_vol(log_returns, window, min_periods=None):
    min_periods = min_periods or max(4, window // 2)
    return log_returns.rolling(window, min_periods=min_periods).std() * np.sqrt(WEEKS_PER_YEAR)


def rolling_downside_semivol(simple_returns, window, min_periods=None):
    min_periods = min_periods or max(4, window // 2)
    downside = simple_returns.clip(upper=0)
    return downside.rolling(window, min_periods=min_periods).std() * np.sqrt(WEEKS_PER_YEAR)


def rolling_drawdown_frequency(prices, window, threshold=-0.05, min_periods=None):
    min_periods = min_periods or max(4, window // 2)
    rolling_high = prices.rolling(window, min_periods=min_periods).max()
    drawdown = prices.div(rolling_high) - 1.0
    return drawdown.lt(threshold).rolling(window, min_periods=min_periods).mean()


def rolling_drawdown_severity(prices, window, min_periods=None):
    min_periods = min_periods or max(4, window // 2)
    rolling_high = prices.rolling(window, min_periods=min_periods).max()
    drawdown = prices.div(rolling_high) - 1.0
    return drawdown.clip(upper=0).abs().rolling(window, min_periods=min_periods).mean()


def rolling_zscore(series, window=MACRO_Z_WINDOW, min_periods=MACRO_MIN_PERIODS):
    mean = series.rolling(window, min_periods=min_periods).mean()
    std = series.rolling(window, min_periods=min_periods).std()
    return (series - mean) / (std + 1e-12)


def winsorize_row(row, lower_q=WINSOR_LOWER_Q, upper_q=WINSOR_UPPER_Q):
    valid = row.dropna()
    out = pd.Series(np.nan, index=row.index, dtype=float)
    if valid.empty:
        return out
    lower = valid.quantile(lower_q)
    upper = valid.quantile(upper_q)
    out.loc[valid.index] = valid.clip(lower, upper)
    return out


def panel_winsorize(panel, lower_q=WINSOR_LOWER_Q, upper_q=WINSOR_UPPER_Q):
    return panel.apply(lambda row: winsorize_row(row, lower_q=lower_q, upper_q=upper_q), axis=1)


def row_rank(row):
    valid = row.dropna()
    out = pd.Series(np.nan, index=row.index, dtype=float)
    if valid.empty:
        return out
    out.loc[valid.index] = valid.rank(method="average")
    return out


def row_normalized_score(row):
    valid = row.dropna()
    out = pd.Series(np.nan, index=row.index, dtype=float)
    if valid.empty:
        return out
    if len(valid) == 1:
        out.loc[valid.index] = 0.0
        return out
    ranks = valid.rank(method="average")
    pct = (ranks - 1.0) / (len(valid) - 1.0)
    out.loc[valid.index] = pct * 2.0 - 1.0
    return out


def row_zscore(row, clip_z=CROSS_SECTIONAL_ZCAP):
    valid = row.dropna()
    out = pd.Series(np.nan, index=row.index, dtype=float)
    if valid.empty:
        return out
    if len(valid) == 1:
        out.loc[valid.index] = 0.0
        return out
    z = (valid - valid.mean()) / (valid.std(ddof=1) + 1e-12)
    out.loc[valid.index] = z.clip(-clip_z, clip_z)
    return out


def row_bucket_labels(row, tail_fraction=0.2):
    valid = row.dropna()
    out = pd.Series(pd.NA, index=row.index, dtype="object")
    if valid.empty:
        return out
    if len(valid) == 1:
        out.loc[valid.index] = "Middle"
        return out
    ranks = valid.rank(method="average")
    pct = (ranks - 1.0) / (len(valid) - 1.0)
    labels = pd.Series("Middle", index=valid.index, dtype="object")
    labels.loc[pct <= tail_fraction] = "Bottom"
    labels.loc[pct >= 1 - tail_fraction] = "Top"
    out.loc[valid.index] = labels
    return out


def panel_rank(panel):
    return panel.apply(row_rank, axis=1)


def panel_score(panel, winsorize=True):
    working = panel_winsorize(panel) if winsorize else panel.copy()
    return working.apply(row_normalized_score, axis=1)


def panel_zscore(panel, winsorize=False):
    working = panel_winsorize(panel) if winsorize else panel.copy()
    return working.apply(row_zscore, axis=1)


def panel_bucket(panel, tail_fraction=0.2):
    return panel.apply(lambda row: row_bucket_labels(row, tail_fraction=tail_fraction), axis=1)


def grouped_panel_score(panel, group_map, min_group_size=ASSET_CLASS_NEUTRAL_MIN_GROUP_SIZE):
    asset_class_series = pd.Series(group_map)
    rows = []
    for _, row in panel.iterrows():
        out = pd.Series(np.nan, index=row.index, dtype=float)
        for asset_class, members in asset_class_series.groupby(asset_class_series):
            tickers = [ticker for ticker in members.index if ticker in row.index]
            subset = row.loc[tickers].dropna()
            if len(subset) < min_group_size:
                continue
            out.loc[subset.index] = row_normalized_score(subset)
        rows.append(out)
    return pd.DataFrame(rows, index=panel.index)


def summarize_panel_coverage(panel, signal_name=None):
    coverage = panel.notna().sum(axis=1)
    return pd.DataFrame(
        [
            {
                "signal_name": signal_name,
                "mean_coverage": coverage.mean(),
                "median_coverage": coverage.median(),
                "min_coverage": coverage.min(),
                "max_coverage": coverage.max(),
            }
        ]
    )


def coverage_by_asset_class(panel, asset_class_map):
    asset_class_series = pd.Series(asset_class_map)
    rows = []
    for asset_class, members in asset_class_series.groupby(asset_class_series):
        tickers = [ticker for ticker in members.index if ticker in panel.columns]
        if not tickers:
            continue
        coverage = panel[tickers].notna().sum(axis=1)
        rows.append(
            {
                "asset_class": asset_class,
                "n_assets": len(tickers),
                "mean_coverage": coverage.mean(),
                "median_coverage": coverage.median(),
            }
        )
    return pd.DataFrame(rows)


def total_lag_weeks(source_type="price"):
    return SIGNAL_LAG_WEEKS + (EXTERNAL_DATA_LAG_WEEKS if source_type == "external" else 0)


def apply_signal_lag(panel, extra_lag_weeks=0):
    return panel.shift(SIGNAL_LAG_WEEKS + extra_lag_weeks)


def compute_forward_returns(simple_returns, horizons):
    gross = 1.0 + simple_returns
    out = {}
    for horizon in horizons:
        product = pd.DataFrame(1.0, index=simple_returns.index, columns=simple_returns.columns)
        for step in range(1, horizon + 1):
            product = product * gross.shift(-step)
        out[horizon] = product - 1.0
    return out


def panel_dict_to_long(panel_map):
    pieces = []
    for name, panel in panel_map.items():
        stacked = panel.stack(dropna=False).rename(name)
        pieces.append(stacked)
    long_df = pd.concat(pieces, axis=1).reset_index()
    long_df = long_df.rename(columns={"level_0": "Date", "level_1": "Ticker"})
    return long_df


def find_first_matching_column(df, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        match = lower_map.get(str(candidate).strip().lower())
        if match is not None:
            return match
    return None


def normalize_ticker_schema(df, preferred="ticker", candidates=("ticker", "Ticker", "symbol", "Symbol"), allow_index=True):
    df = pd.DataFrame() if df is None else df.copy()
    if preferred in df.columns:
        return df
    match = find_first_matching_column(df, candidates)
    if match is not None:
        if match != preferred:
            df = df.rename(columns={match: preferred})
        return df
    if allow_index and not isinstance(df.index, pd.RangeIndex):
        index_name = str(df.index.name).strip().lower() if df.index.name is not None else ""
        candidate_names = {str(candidate).strip().lower() for candidate in candidates} | {str(preferred).strip().lower()}
        if index_name in candidate_names or (not index_name and len(df.index) > 0):
            df = df.reset_index()
            first_col = df.columns[0]
            if first_col != preferred:
                df = df.rename(columns={first_col: preferred})
    if preferred not in df.columns:
        df[preferred] = pd.Series(dtype="object")
    return df


def rolling_beta_panel(asset_returns, market_returns, window=BETA_WINDOW, min_periods=BETA_MIN_PERIODS):
    market_var = market_returns.rolling(window, min_periods=min_periods).var()
    beta_parts = {}
    for ticker in asset_returns.columns:
        cov = asset_returns[ticker].rolling(window, min_periods=min_periods).cov(market_returns)
        beta_parts[ticker] = cov.div(market_var)
    beta = pd.DataFrame(beta_parts)
    beta.index = asset_returns.index
    return beta[asset_returns.columns]


def compute_residual_log_returns(asset_log_returns, market_log_return, window=RESIDUAL_BETA_WINDOW, min_periods=BETA_MIN_PERIODS):
    beta = rolling_beta_panel(asset_log_returns, market_log_return, window=window, min_periods=min_periods)
    market_mean = market_log_return.rolling(window, min_periods=min_periods).mean()
    alpha = asset_log_returns.rolling(window, min_periods=min_periods).mean().sub(beta.mul(market_mean, axis=0), axis=0)
    expected = alpha.shift(1).add(beta.shift(1).mul(market_log_return, axis=0), axis=0)
    residual = asset_log_returns - expected
    return residual, beta.shift(1), alpha.shift(1)


def safe_ticker_info(ticker):
    snapshot = {"ticker": ticker, "pull_timestamp_utc": pd.Timestamp.utcnow().isoformat()}
    if yf is None:
        snapshot["ticker_error"] = "yfinance not installed"
        return snapshot
    try:
        tk = yf.Ticker(ticker)
    except Exception as e:
        snapshot["ticker_error"] = str(e)
        return snapshot
    try:
        info = tk.info or {}
    except Exception as e:
        info = {}
        snapshot["info_error"] = str(e)
    for field in [
        "shortName",
        "longName",
        "quoteType",
        "category",
        "fundFamily",
        "currency",
        "yield",
        "dividendYield",
        "trailingAnnualDividendYield",
        "expenseRatio",
        "beta3Year",
        "totalAssets",
    ]:
        snapshot[field] = info.get(field)
    return snapshot


def safe_dividend_history(ticker):
    if yf is None:
        return pd.Series(dtype=float, name=ticker)
    try:
        dividends = yf.Ticker(ticker).dividends
        if dividends is None or len(dividends) == 0:
            return pd.Series(dtype=float, name=ticker)
        return ensure_series(dividends, name=ticker)
    except Exception:
        return pd.Series(dtype=float, name=ticker)


def snapshot_yahoo_metadata(tickers):
    records = []
    for ticker in tickers:
        records.append(safe_ticker_info(ticker))
        time.sleep(0.1)
    return pd.DataFrame(records)


def snapshot_distribution_history(tickers):
    frames = []
    for ticker in tickers:
        dividends = safe_dividend_history(ticker)
        if dividends.empty:
            continue
        frame = dividends.reset_index()
        frame.columns = ["Date", "distribution"]
        frame["ticker"] = ticker
        frame["pull_timestamp_utc"] = pd.Timestamp.utcnow().isoformat()
        frames.append(frame)
        time.sleep(0.1)
    if not frames:
        return pd.DataFrame(columns=["Date", "ticker", "distribution", "pull_timestamp_utc"])
    history = pd.concat(frames, ignore_index=True)
    history["Date"] = pd.to_datetime(history["Date"]).dt.strftime("%Y-%m-%d")
    return history


def build_carry_proxy(
    tickers,
    weekly_prices,
    distribution_history,
    metadata_snapshot,
    trailing_weeks=CARRY_TRAILING_WEEKS,
    live_fallback=False,
):
    dist = normalize_ticker_schema(distribution_history, preferred="ticker")
    if not dist.empty:
        date_col = find_first_matching_column(dist, ("Date", "date", "datetime", "timestamp"))
        value_col = find_first_matching_column(dist, ("distribution", "dividend", "dividends"))
        if date_col and "ticker" in dist.columns and value_col:
            dist = dist[[date_col, "ticker", value_col]].copy()
            dist.columns = ["Date", "ticker", "distribution"]
            dist["Date"] = pd.to_datetime(dist["Date"], errors="coerce")
            dist["ticker"] = dist["ticker"].astype(str).str.strip().replace({"": np.nan})
            dist["distribution"] = pd.to_numeric(dist["distribution"], errors="coerce")
            dist = dist.dropna(subset=["Date", "ticker", "distribution"])
        else:
            dist = pd.DataFrame(columns=["Date", "ticker", "distribution"])

    source_label = "cached_distribution_history"

    if dist.empty and live_fallback and yf is not None:
        dist = normalize_ticker_schema(snapshot_distribution_history(tickers), preferred="ticker")
        if not dist.empty:
            date_col = find_first_matching_column(dist, ("Date", "date", "datetime", "timestamp"))
            value_col = find_first_matching_column(dist, ("distribution", "dividend", "dividends"))
            if date_col and "ticker" in dist.columns and value_col:
                dist = dist[[date_col, "ticker", value_col]].copy()
                dist.columns = ["Date", "ticker", "distribution"]
                dist["Date"] = pd.to_datetime(dist["Date"], errors="coerce")
                dist["ticker"] = dist["ticker"].astype(str).str.strip().replace({"": np.nan})
                dist["distribution"] = pd.to_numeric(dist["distribution"], errors="coerce")
                dist = dist.dropna(subset=["Date", "ticker", "distribution"])
                source_label = "live_distribution_history"
            else:
                dist = pd.DataFrame(columns=["Date", "ticker", "distribution"])

    if dist.empty:
        yield_panel = pd.DataFrame(np.nan, index=weekly_prices.index, columns=tickers)
    else:
        pivot = (
            dist.pivot_table(index="Date", columns="ticker", values="distribution", aggfunc="sum")
            .sort_index()
        )
        weekly_dist = pivot.resample("W-FRI").sum().reindex(weekly_prices.index, fill_value=0.0)
        yield_panel = weekly_dist.rolling(trailing_weeks, min_periods=4).sum().div(
            weekly_prices.reindex(columns=weekly_dist.columns)
        )
        yield_panel = yield_panel.reindex(index=weekly_prices.index, columns=tickers)
    yield_panel.index.name = "Date"
    yield_panel.columns.name = None

    meta = normalize_ticker_schema(metadata_snapshot, preferred="ticker")
    if not meta.empty:
        meta.columns = [str(c).strip().lower() for c in meta.columns]
        meta = normalize_ticker_schema(meta, preferred="ticker", candidates=("ticker", "symbol"), allow_index=True)
        if "ticker" in meta.columns:
            meta["ticker"] = meta["ticker"].astype(str).str.strip().replace({"": np.nan})
            meta = meta.dropna(subset=["ticker"]).drop_duplicates(subset=["ticker"], keep="last").set_index("ticker")

    source_rows = []
    for ticker in tickers:
        distribution_rows = 0 if dist.empty else int((dist["ticker"] == ticker).sum())
        metadata_yield_latest = np.nan
        if not meta.empty and ticker in meta.index:
            row = meta.loc[ticker]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[-1]
            for field in ["yield", "dividendyield", "trailingannualdividendyield"]:
                if field in row.index and pd.notna(row[field]):
                    metadata_yield_latest = pd.to_numeric(row[field], errors="coerce")
                    break
        if distribution_rows > 0:
            source = source_label
        elif pd.notna(metadata_yield_latest):
            source = "metadata_snapshot_only"
        else:
            source = "missing"
        source_rows.append(
            {
                "ticker": ticker,
                "carry_source": source,
                "distribution_rows": distribution_rows,
                "metadata_yield_latest": metadata_yield_latest,
            }
        )

    return yield_panel, normalize_ticker_schema(pd.DataFrame(source_rows), preferred="ticker", allow_index=False)


def select_market_proxy_log_return(market_proxy_weekly, benchmark_returns_weekly, weekly_log_returns, proxy_mapping):
    market_proxy_ticker = get_proxy_ticker(proxy_mapping, "market_proxy", fallback="SPY")

    if not market_proxy_weekly.empty and "market_proxy_log_return" in market_proxy_weekly.columns:
        return market_proxy_weekly["market_proxy_log_return"], market_proxy_ticker, "market_proxy_weekly.csv"

    if not benchmark_returns_weekly.empty and market_proxy_ticker in benchmark_returns_weekly.columns:
        return benchmark_returns_weekly[market_proxy_ticker], market_proxy_ticker, "benchmark_returns_weekly.csv"

    if market_proxy_ticker in weekly_log_returns.columns:
        return weekly_log_returns[market_proxy_ticker], market_proxy_ticker, "weekly_returns.csv"

    raise ValueError("No market proxy log-return series is available for BAB / residual momentum.")


def plot_signal_examples(panel, title, tickers=("SPY", "TLT", "GLD"), lookback_weeks=260, ax=None):
    ax = ax or plt.gca()
    subset = [ticker for ticker in tickers if ticker in panel.columns]
    if not subset:
        ax.set_title(f"{title}\n(no example tickers found)")
        return ax
    panel[subset].iloc[-lookback_weeks:].plot(ax=ax, lw=1.5)
    ax.set_title(title)
    ax.set_xlabel("")
    return ax


In [ ]:
def download_daily_prices(universe, start_date, end_date=None):
    if yf is None:
        raise ImportError("yfinance is required to rebuild missing upstream inputs.")
    raw_data = {}
    failed = []
    for ticker in universe:
        try:
            series = safe_download_close(ticker, start=start_date, end=end_date, auto_adjust=True)
            if series is None or series.dropna().empty:
                failed.append(ticker)
            else:
                raw_data[ticker] = series
        except Exception:
            failed.append(ticker)
        time.sleep(0.15)
    if not raw_data:
        raise ValueError("No price history could be downloaded for the fallback rebuild.")
    daily_prices = pd.concat(raw_data.values(), axis=1)
    daily_prices = flatten_columns(daily_prices).sort_index(axis=1)
    return daily_prices, failed


def build_weekly_price_inputs(universe, start_date, end_date=None):
    daily_prices, failed = download_daily_prices(universe, start_date, end_date=end_date)
    daily_log_returns = np.log(daily_prices / daily_prices.shift(1)).dropna(how="all")
    daily_log_returns = flatten_columns(daily_log_returns)
    weekly_prices = daily_prices.resample("W-FRI").last()
    weekly_prices = flatten_columns(weekly_prices)
    last_daily_date = daily_prices.index.max()
    last_weekly_label = weekly_prices.index.max()
    if last_daily_date < last_weekly_label - pd.Timedelta(days=1):
        weekly_prices = weekly_prices.iloc[:-1]
    weekly_log_returns = np.log(weekly_prices / weekly_prices.shift(1)).dropna(how="all")
    weekly_log_returns = flatten_columns(weekly_log_returns)
    return daily_prices, daily_log_returns, weekly_prices, weekly_log_returns, failed


def build_macro_weekly(weekly_index, start_date, fred_series, fred_api_key=None):
    macro_raw = {}
    if Fred is not None and fred_series:
        try:
            fred = Fred(api_key=fred_api_key) if fred_api_key else Fred()
            for series_id in fred_series:
                try:
                    series = fred.get_series(series_id, observation_start=start_date)
                    macro_raw[series_id] = ensure_series(series, name=series_id)
                except Exception:
                    continue
        except Exception:
            pass
    if not macro_raw:
        macro = pd.DataFrame(index=weekly_index)
        macro.index.name = "Date"
        return macro
    macro_df = pd.concat(macro_raw.values(), axis=1)
    macro_df = flatten_columns(macro_df)
    macro_weekly = macro_df.resample("W-FRI").last().ffill()
    macro_weekly = macro_weekly.reindex(weekly_index, method="ffill")
    if {"FEDFUNDS", "DGS3MO"}.issubset(macro_weekly.columns):
        macro_weekly["policy_minus_3m"] = macro_weekly["FEDFUNDS"] - macro_weekly["DGS3MO"]
    return flatten_columns(macro_weekly)


def build_vix_term_structure(weekly_index, start_date, end_date=None):
    if yf is None:
        raise ImportError("yfinance is required to rebuild the VIX term structure.")
    vix_parts = []
    for ticker in ["^VIX", "^VIX3M", "^VIX6M"]:
        series = safe_download_close(ticker, start=start_date, end=end_date, auto_adjust=False)
        if series is not None:
            vix_parts.append(series)
    if not vix_parts:
        return pd.DataFrame(index=weekly_index)
    vix_df = pd.concat(vix_parts, axis=1)
    vix_df = flatten_columns(vix_df)
    vix_df = vix_df.rename(columns={"^VIX": "VIX", "^VIX3M": "VIX3M", "^VIX6M": "VIX6M"})
    vix_weekly = vix_df.resample("W-FRI").last().reindex(weekly_index, method="ffill")
    vix_weekly["slope_1m_3m"] = vix_weekly.get("VIX3M") - vix_weekly.get("VIX")
    if {"VIX6M", "VIX"}.issubset(vix_weekly.columns):
        vix_weekly["slope_1m_6m"] = vix_weekly["VIX6M"] - vix_weekly["VIX"]
    vix_weekly["contango"] = (vix_weekly["slope_1m_3m"] > 0).astype(float)
    return flatten_columns(vix_weekly)


def build_google_trends(weekly_index, start_date):
    raw = pd.DataFrame()
    processed = pd.DataFrame(index=weekly_index)
    processed.index.name = "Date"
    meta = {
        "pull_timestamp_utc": pd.Timestamp.utcnow().isoformat(),
        "keywords": ["recession", "stock market crash", "inflation", "bear market"],
        "status": "not_run",
    }

    if TrendReq is None:
        processed["fear_composite"] = np.nan
        processed["fear_composite_zscore"] = np.nan
        meta["status"] = "pytrends_not_installed"
        return raw, processed, meta

    keywords = meta["keywords"]
    pytrends = TrendReq(hl="en-US", tz=360, timeout=(10, 25))
    frames = []
    for keyword in keywords:
        try:
            pytrends.build_payload(
                [keyword],
                cat=0,
                timeframe=f"{start_date} {pd.Timestamp.today().strftime('%Y-%m-%d')}",
                geo="US",
            )
            data = pytrends.interest_over_time()
            if not data.empty and keyword in data.columns:
                frames.append(ensure_series(data[keyword], name=keyword).resample("W-FRI").mean())
        except Exception:
            continue
        time.sleep(2)

    if not frames:
        processed["fear_composite"] = np.nan
        processed["fear_composite_zscore"] = np.nan
        meta["status"] = "no_data"
        return raw, processed, meta

    raw = flatten_columns(pd.concat(frames, axis=1))
    processed = raw.resample("W-FRI").last().reindex(weekly_index).ffill(limit=2)
    for col in list(processed.columns):
        processed[f"{col}_zscore"] = rolling_zscore(processed[col], window=MACRO_Z_WINDOW, min_periods=MACRO_MIN_PERIODS)
    raw_cols = [c for c in processed.columns if "_zscore" not in c]
    processed["fear_composite"] = processed[raw_cols].mean(axis=1)
    processed["fear_composite_zscore"] = rolling_zscore(processed["fear_composite"], window=MACRO_Z_WINDOW, min_periods=MACRO_MIN_PERIODS)
    processed = flatten_columns(processed)
    meta["status"] = "success"
    return raw, processed, meta


def build_quality_and_universe(weekly_prices, universe_map, asset_class_map, proxy_mapping):
    quality_records = []
    for ticker in weekly_prices.columns:
        price_series = weekly_prices[ticker].dropna()
        n_total = len(weekly_prices)
        if price_series.empty:
            quality_records.append(
                {
                    "Ticker": ticker,
                    "Description": universe_map.get(ticker, "Unknown"),
                    "Asset Class": asset_class_map.get(ticker, "Unknown"),
                    "Start Date": None,
                    "End Date": None,
                    "Weeks Available": 0,
                    "Weeks Missing": n_total,
                    "Completeness %": 0.0,
                    "Has Full History": False,
                    "Ann Vol %": np.nan,
                }
            )
            continue
        rets = np.log(price_series / price_series.shift(1)).dropna()
        ann_vol = rets.std() * np.sqrt(WEEKS_PER_YEAR) * 100
        quality_records.append(
            {
                "Ticker": ticker,
                "Description": universe_map.get(ticker, "Unknown"),
                "Asset Class": asset_class_map.get(ticker, "Unknown"),
                "Start Date": str(price_series.index.min().date()),
                "End Date": str(price_series.index.max().date()),
                "Weeks Available": int(price_series.notna().sum()),
                "Weeks Missing": int(n_total - price_series.notna().sum()),
                "Completeness %": round(price_series.notna().sum() / n_total * 100, 1),
                "Has Full History": bool((n_total - price_series.notna().sum()) <= 10),
                "Ann Vol %": round(float(ann_vol), 1),
            }
        )
    quality = pd.DataFrame(quality_records).sort_values("Weeks Available", ascending=False)
    full_universe = quality.loc[quality["Weeks Available"] >= 156, "Ticker"].tolist()
    core_universe = quality.loc[quality["Weeks Available"] >= 520, "Ticker"].tolist()
    universe_def = {
        "core": core_universe,
        "full": full_universe,
        "all": list(weekly_prices.columns),
        "descriptions": universe_map,
        "asset_class_map": asset_class_map,
        "proxy_mapping": proxy_mapping,
    }
    universe_metadata = quality.rename(
        columns={"Ticker": "ticker", "Description": "description", "Asset Class": "asset_class"}
    ).copy()
    universe_metadata["in_core_universe"] = universe_metadata["ticker"].isin(core_universe)
    universe_metadata["in_full_universe"] = universe_metadata["ticker"].isin(full_universe)
    return quality, universe_def, universe_metadata


def save_proxy_files(target_dir, weekly_prices, weekly_log_returns, proxy_mapping):
    proxy_mapping = proxy_mapping or {}
    benchmark_rows = []
    benchmark_tickers = []
    for role, info in proxy_mapping.items():
        ticker = get_proxy_ticker(proxy_mapping, role)
        benchmark_rows.append(
            {
                "proxy_role": role,
                "ticker": ticker,
                "description": info.get("description") if isinstance(info, dict) else role,
                "available_in_universe": ticker in weekly_prices.columns,
            }
        )
        if ticker in weekly_prices.columns:
            benchmark_tickers.append(ticker)

    pd.DataFrame(benchmark_rows).to_csv(target_dir / "benchmarks.csv", index=False)
    (target_dir / "proxy_mapping.json").write_text(json.dumps(proxy_mapping, indent=2))

    benchmark_tickers = sorted(set(benchmark_tickers))
    weekly_prices.reindex(columns=benchmark_tickers).to_csv(target_dir / "benchmark_prices_weekly.csv")
    weekly_log_returns.reindex(columns=benchmark_tickers).to_csv(target_dir / "benchmark_returns_weekly.csv")

    market_proxy_ticker = get_proxy_ticker(proxy_mapping, "market_proxy", fallback="SPY")
    if market_proxy_ticker in weekly_prices.columns and market_proxy_ticker in weekly_log_returns.columns:
        market_proxy_weekly = pd.DataFrame(
            {
                "market_proxy_price": weekly_prices[market_proxy_ticker],
                "market_proxy_log_return": weekly_log_returns[market_proxy_ticker],
            }
        )
        market_proxy_weekly.to_csv(target_dir / "market_proxy_weekly.csv")


def rebuild_required_inputs(target_dir, required_files):
    if not DATA_HUB_PATH.exists():
        raise FileNotFoundError("Cannot rebuild inputs because 01_data_hub.ipynb is missing.")

    universe_map = extract_literal_assignment(DATA_HUB_PATH, "UNIVERSE") or {}
    asset_class_map = extract_literal_assignment(DATA_HUB_PATH, "UNIVERSE_ASSET_CLASS") or {}
    proxy_mapping = extract_literal_assignment(DATA_HUB_PATH, "PROXY_MAPPING") or {
        "market_proxy": {"ticker": "SPY", "description": "Fallback market proxy"},
        "cash_proxy": {"ticker": "BIL", "description": "Fallback cash proxy"},
        "duration_proxy": {"ticker": "TLT", "description": "Fallback duration proxy"},
    }
    start_date = extract_literal_assignment(DATA_HUB_PATH, "START_DATE") or "2005-01-01"
    end_date = extract_literal_assignment(DATA_HUB_PATH, "END_DATE")
    fred_series = extract_literal_assignment(DATA_HUB_PATH, "FRED_SERIES") or {}
    fred_api_key = extract_literal_assignment(DATA_HUB_PATH, "FRED_API_KEY")

    if not universe_map:
        raise ValueError("Could not extract UNIVERSE from 01_data_hub.ipynb for fallback rebuild.")

    tickers = list(universe_map.keys())
    for ticker in tickers:
        asset_class_map.setdefault(ticker, infer_asset_class_from_description(universe_map.get(ticker)))

    print("Rebuilding missing upstream inputs from notebook conventions...")
    print(f"  Start date: {start_date}")
    print(f"  Universe size: {len(tickers)}")

    daily_prices, daily_log_returns, weekly_prices, weekly_log_returns, failed = build_weekly_price_inputs(
        tickers,
        start_date,
        end_date=end_date,
    )
    daily_prices.to_csv(target_dir / "daily_prices.csv")
    daily_log_returns.to_csv(target_dir / "daily_returns.csv")
    weekly_prices.to_csv(target_dir / "weekly_prices.csv")
    weekly_log_returns.to_csv(target_dir / "weekly_returns.csv")

    macro_weekly = build_macro_weekly(weekly_prices.index, start_date, fred_series, fred_api_key=fred_api_key)
    macro_weekly.to_csv(target_dir / "macro_weekly.csv")

    vix_weekly = build_vix_term_structure(weekly_prices.index, start_date, end_date=end_date)
    vix_weekly.to_csv(target_dir / "vix_term_structure.csv")

    google_trends_raw, trends_weekly, trends_meta = build_google_trends(weekly_prices.index, start_date)
    trends_weekly.to_csv(target_dir / "google_trends.csv")
    google_trends_raw.to_csv(target_dir / "google_trends_raw.csv")
    (target_dir / "google_trends_snapshot_meta.json").write_text(json.dumps(trends_meta, indent=2))

    quality_df, universe_def, universe_metadata = build_quality_and_universe(
        weekly_prices,
        universe_map,
        asset_class_map,
        proxy_mapping,
    )
    quality_df.to_csv(target_dir / "data_quality_report.csv", index=False)
    (target_dir / "universe.json").write_text(json.dumps(universe_def, indent=2))
    universe_metadata.to_csv(target_dir / "universe_metadata.csv", index=False)

    save_proxy_files(target_dir, weekly_prices, weekly_log_returns, proxy_mapping)

    metadata_snapshot = snapshot_yahoo_metadata(tickers)
    metadata_snapshot.to_csv(target_dir / "yahoo_metadata_snapshot.csv", index=False)
    distribution_history = snapshot_distribution_history(tickers)
    distribution_history.to_csv(target_dir / "etf_distribution_history.csv", index=False)

    if failed:
        print("Failed tickers during fallback rebuild:", failed)

    still_missing = [name for name in required_files if not (target_dir / name).exists()]
    if still_missing:
        raise FileNotFoundError(f"Fallback rebuild completed, but files are still missing: {still_missing}")


In [ ]:
input_dir, missing_inputs = choose_input_dir(INPUT_DIR_CANDIDATES, REQUIRED_INPUTS)

print("Best upstream input directory guess:", input_dir.resolve() if input_dir else None)
print("Missing required upstream files:", missing_inputs)

if missing_inputs:
    if AUTO_REBUILD_MISSING_INPUTS:
        rebuild_required_inputs(DATA_HUB_DIR, REQUIRED_INPUTS)
        input_dir = DATA_HUB_DIR
    else:
        raise FileNotFoundError(
            "Required processed inputs are missing. "
            "Run 01_data_hub.ipynb first or set AUTO_REBUILD_MISSING_INPUTS = True."
        )

weekly_prices = read_panel_csv(input_dir / "weekly_prices.csv")
weekly_log_returns = read_panel_csv(input_dir / "weekly_returns.csv")
daily_log_returns = read_panel_csv(input_dir / "daily_returns.csv")
macro_weekly = read_panel_csv(input_dir / "macro_weekly.csv")
vix_term = read_panel_csv(input_dir / "vix_term_structure.csv")
google_trends = read_panel_csv(input_dir / "google_trends.csv")
benchmark_prices_weekly = read_panel_csv(input_dir / "benchmark_prices_weekly.csv")
benchmark_returns_weekly = read_panel_csv(input_dir / "benchmark_returns_weekly.csv")
market_proxy_weekly = read_panel_csv(input_dir / "market_proxy_weekly.csv")

quality_report = read_table_csv(input_dir / "data_quality_report.csv")
universe_def = read_json_file(input_dir / "universe.json")
proxy_mapping = read_json_file(input_dir / "proxy_mapping.json")
universe_metadata = read_table_csv(input_dir / "universe_metadata.csv")
yahoo_metadata_snapshot = read_table_csv(input_dir / "yahoo_metadata_snapshot.csv")
distribution_history = read_table_csv(input_dir / "etf_distribution_history.csv")
google_trends_raw = read_panel_csv(input_dir / "google_trends_raw.csv")

if weekly_prices.empty or weekly_log_returns.empty:
    raise ValueError("Weekly prices / returns could not be loaded.")

research_universe = universe_def.get("full", universe_def.get("all", list(weekly_prices.columns)))
research_universe = [ticker for ticker in research_universe if ticker in weekly_prices.columns]

asset_class_map = build_asset_class_map(universe_def, universe_metadata, research_universe)
asset_class_series = pd.Series(asset_class_map).reindex(research_universe)
asset_class_counts = asset_class_series.value_counts()
asset_class_frame = pd.DataFrame({"Ticker": research_universe, "asset_class": asset_class_series.values})

market_proxy_log_return, market_proxy_ticker, market_proxy_source = select_market_proxy_log_return(
    market_proxy_weekly,
    benchmark_returns_weekly,
    weekly_log_returns,
    proxy_mapping,
)

if market_proxy_ticker in weekly_prices.columns and market_proxy_ticker not in research_universe:
    research_universe = sorted(set(research_universe + [market_proxy_ticker]))
    asset_class_map[market_proxy_ticker] = asset_class_map.get(market_proxy_ticker, "Equities")
    asset_class_series = pd.Series(asset_class_map).reindex(research_universe)
    asset_class_counts = asset_class_series.value_counts()
    asset_class_frame = pd.DataFrame({"Ticker": research_universe, "asset_class": asset_class_series.values})

weekly_prices = weekly_prices.reindex(columns=research_universe)
weekly_log_returns = weekly_log_returns.reindex(index=weekly_prices.index, columns=weekly_prices.columns)
if not daily_log_returns.empty:
    daily_log_returns = daily_log_returns.reindex(columns=research_universe)

weekly_simple_returns = weekly_prices.div(weekly_prices.shift(1)) - 1.0
weekly_simple_from_log = np.expm1(weekly_log_returns)
weekly_simple_returns = weekly_simple_returns.where(~weekly_log_returns.isna())

recomputed_log = np.log(weekly_prices / weekly_prices.shift(1)).reindex_like(weekly_log_returns)
max_log_return_diff = (recomputed_log - weekly_log_returns).abs().max().max()
max_simple_diff = (weekly_simple_returns - weekly_simple_from_log).abs().max().max()
week_days = sorted(pd.Index(weekly_prices.index).weekday.unique().tolist())

available_optional_inputs = [name for name in OPTIONAL_INPUTS if (input_dir / name).exists()]

sanity_table = pd.DataFrame(
    [
        {"check": "Weekly index uses Friday labels", "result": week_days == [4], "detail": str(week_days)},
        {"check": "Stored log returns match price-derived log returns", "result": bool(max_log_return_diff < 1e-8), "detail": float(max_log_return_diff)},
        {"check": "Simple returns from prices match exp(log_returns)-1", "result": bool(max_simple_diff < 1e-8), "detail": float(max_simple_diff)},
        {"check": "Explicit market proxy available", "result": True, "detail": f"{market_proxy_ticker} via {market_proxy_source}"},
        {"check": "Signal lag weeks", "result": True, "detail": SIGNAL_LAG_WEEKS},
        {"check": "External feature extra lag weeks", "result": True, "detail": EXTERNAL_DATA_LAG_WEEKS},
    ]
)

print("Weekly data audit")
print("=" * 80)
print(sanity_table.to_string(index=False))
print()
print("Optional upstream artifacts found:")
print(sorted(available_optional_inputs))
print()
print("Asset-class counts")
print(asset_class_counts.to_string())

if not quality_report.empty:
    quality_cols_lower = {str(c).lower(): c for c in quality_report.columns}
    if "ticker" in quality_cols_lower:
        quality_report = quality_report.set_index(quality_cols_lower["ticker"])

if not macro_weekly.empty:
    macro_weekly = macro_weekly.reindex(weekly_prices.index).ffill()
if not vix_term.empty:
    vix_term = vix_term.reindex(weekly_prices.index).ffill()
if not google_trends.empty:
    google_trends = google_trends.reindex(weekly_prices.index).ffill(limit=2)

forward_returns = compute_forward_returns(weekly_simple_returns, FORWARD_HORIZONS)
one_week_ahead_returns = forward_returns[1]

print()
print(f"Research universe size: {len(research_universe)}")
print(f"Sample dates: {weekly_prices.index.min().date()} -> {weekly_prices.index.max().date()}")
print(f"Signal outputs will be written to: {OUTPUT_DIR.resolve()}")


## 2. Validation Framework

The goal is not to “prove” any one signal with one chart. The goal is to stress-test the signal from several angles while respecting the timing convention defined above.

**For cross-sectional signals**

- compute forward returns at 1, 2, 4, 8, and 12 weeks
- measure cross-sectional Spearman IC by date
- summarize mean IC, IC standard deviation, ordinary t-stat, optional Newey-West style t-stat, hit rate, and coverage

**For time-series signals**

We use a pooled ETF implementation that asks a simpler question:

- when the signal is positive, are future returns usually better than when it is negative?
- does a sign-based strategy have positive average signed return and a reasonable t-stat?

**Implementation realism**

Every tradable cross-sectional score also gets:

- a simple market-neutral weekly weight construction
- a turnover proxy
- a cost-sensitivity table at 0, 5, 10, and 25 bps

**Autocorrelation caveat**

Ordinary t-stats can look too strong when IC or signal returns are autocorrelated.
This notebook therefore reports a lightweight Newey-West style t-stat as an additional robustness check when enough observations exist.

This is still not a full inferential treatment, but it is better than treating every weekly observation as fully independent.


In [ ]:
def safe_t_stat(series):
    series = pd.Series(series).dropna()
    if len(series) < 2:
        return np.nan
    std = series.std(ddof=1)
    if std == 0 or pd.isna(std):
        return np.nan
    return series.mean() / (std / np.sqrt(len(series)))


def newey_west_tstat(series, max_lags=NEWEY_WEST_MAX_LAGS):
    series = pd.Series(series).dropna().astype(float)
    n = len(series)
    if n < 3:
        return np.nan
    if max_lags is None:
        max_lags = max(1, int(np.floor(4 * (n / 100) ** (2 / 9))))
    max_lags = min(max_lags, n - 1)
    centered = series - series.mean()
    gamma0 = float(np.dot(centered, centered) / n)
    long_run_var = gamma0
    for lag in range(1, max_lags + 1):
        cov = float(np.dot(centered.iloc[lag:], centered.iloc[:-lag]) / n)
        weight = 1.0 - lag / (max_lags + 1.0)
        long_run_var += 2.0 * weight * cov
    if long_run_var <= 0:
        return np.nan
    se_mean = np.sqrt(long_run_var / n)
    if se_mean == 0 or pd.isna(se_mean):
        return np.nan
    return series.mean() / se_mean


def evaluate_cross_sectional_signal(signal_name, score_panel, forward_returns, min_assets=MIN_CROSS_SECTION):
    summary_rows = []
    ic_rows = []
    for horizon, fwd_panel in forward_returns.items():
        horizon_rows = []
        for date in score_panel.index:
            joint = pd.concat(
                [score_panel.loc[date].rename("signal"), fwd_panel.loc[date].rename("forward_return")],
                axis=1,
            ).dropna()
            if len(joint) < min_assets:
                continue
            ic = joint["signal"].corr(joint["forward_return"], method="spearman")
            horizon_rows.append(
                {
                    "Date": date,
                    "signal_name": signal_name,
                    "horizon_weeks": horizon,
                    "ic": ic,
                    "coverage": len(joint),
                }
            )
        horizon_df = pd.DataFrame(horizon_rows)
        if horizon_df.empty:
            summary_rows.append(
                {
                    "signal_name": signal_name,
                    "evaluation_type": "cross_sectional",
                    "horizon_weeks": horizon,
                    "mean_ic": np.nan,
                    "ic_std": np.nan,
                    "ic_tstat": np.nan,
                    "ic_tstat_nw": np.nan,
                    "hit_rate": np.nan,
                    "mean_coverage": np.nan,
                    "median_coverage": np.nan,
                    "n_dates": 0,
                }
            )
            continue
        ic_rows.append(horizon_df)
        summary_rows.append(
            {
                "signal_name": signal_name,
                "evaluation_type": "cross_sectional",
                "horizon_weeks": horizon,
                "mean_ic": horizon_df["ic"].mean(),
                "ic_std": horizon_df["ic"].std(ddof=1),
                "ic_tstat": safe_t_stat(horizon_df["ic"]),
                "ic_tstat_nw": newey_west_tstat(horizon_df["ic"]) if USE_NEWEY_WEST_TSTATS else np.nan,
                "hit_rate": (horizon_df["ic"] > 0).mean(),
                "mean_coverage": horizon_df["coverage"].mean(),
                "median_coverage": horizon_df["coverage"].median(),
                "n_dates": len(horizon_df),
            }
        )
    ic_ts = pd.concat(ic_rows, ignore_index=True) if ic_rows else pd.DataFrame()
    return pd.DataFrame(summary_rows), ic_ts


def evaluate_time_series_signal(signal_name, signal_panel, forward_returns):
    summary_rows = []
    bucket_rows = []
    for horizon, fwd_panel in forward_returns.items():
        stacked = pd.concat(
            [signal_panel.stack(dropna=False).rename("signal"), fwd_panel.stack(dropna=False).rename("forward_return")],
            axis=1,
        ).dropna()
        if stacked.empty:
            summary_rows.append(
                {
                    "signal_name": signal_name,
                    "evaluation_type": "time_series",
                    "horizon_weeks": horizon,
                    "directional_hit_rate": np.nan,
                    "mean_forward_when_positive": np.nan,
                    "mean_forward_when_negative": np.nan,
                    "signed_strategy_mean": np.nan,
                    "signed_strategy_tstat": np.nan,
                    "signed_strategy_tstat_nw": np.nan,
                    "coverage_obs": 0,
                }
            )
            continue

        direction = np.sign(stacked["signal"])
        nonzero = direction != 0
        signed_strategy = direction[nonzero] * stacked.loc[nonzero, "forward_return"]
        summary_rows.append(
            {
                "signal_name": signal_name,
                "evaluation_type": "time_series",
                "horizon_weeks": horizon,
                "directional_hit_rate": (np.sign(stacked.loc[nonzero, "forward_return"]) == direction[nonzero]).mean(),
                "mean_forward_when_positive": stacked.loc[stacked["signal"] > 0, "forward_return"].mean(),
                "mean_forward_when_negative": stacked.loc[stacked["signal"] < 0, "forward_return"].mean(),
                "signed_strategy_mean": signed_strategy.mean(),
                "signed_strategy_tstat": safe_t_stat(signed_strategy),
                "signed_strategy_tstat_nw": newey_west_tstat(signed_strategy) if USE_NEWEY_WEST_TSTATS else np.nan,
                "coverage_obs": len(stacked),
            }
        )

        if stacked["signal"].nunique() >= 5:
            temp = stacked.copy()
            temp["bucket"] = pd.qcut(temp["signal"], q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"], duplicates="drop")
            grouped = temp.groupby("bucket", observed=False)["forward_return"].mean().reset_index()
            grouped["signal_name"] = signal_name
            grouped["horizon_weeks"] = horizon
            bucket_rows.append(grouped)

    bucket_table = pd.concat(bucket_rows, ignore_index=True) if bucket_rows else pd.DataFrame()
    return pd.DataFrame(summary_rows), bucket_table


def build_market_neutral_weights(score_panel):
    weights = []
    for _, row in score_panel.iterrows():
        valid = row.dropna()
        out = pd.Series(0.0, index=row.index, dtype=float)
        if len(valid) >= 2:
            centered = valid - valid.mean()
            gross = centered.abs().sum()
            if gross > 0:
                out.loc[valid.index] = centered / gross
        weights.append(out)
    weights_df = pd.DataFrame(weights, index=score_panel.index)
    weights_df.columns = score_panel.columns
    return weights_df


def transaction_cost_table(signal_name, score_panel, one_week_ahead_returns, cost_bps_grid=COST_BPS_GRID):
    weights = build_market_neutral_weights(score_panel)
    gross_returns = (weights * one_week_ahead_returns).sum(axis=1)
    turnover = 0.5 * weights.fillna(0.0).diff().abs().sum(axis=1)
    turnover.iloc[0] = np.nan

    rows = []
    for cost_bps in cost_bps_grid:
        cost = turnover.fillna(0.0) * (cost_bps / 10000.0)
        net_returns = gross_returns - cost
        vol = net_returns.std(ddof=1)
        sharpe = np.nan if pd.isna(vol) or vol == 0 else (net_returns.mean() / vol) * np.sqrt(WEEKS_PER_YEAR)
        rows.append(
            {
                "signal_name": signal_name,
                "cost_bps": cost_bps,
                "mean_weekly_return": net_returns.mean(),
                "ann_return": net_returns.mean() * WEEKS_PER_YEAR,
                "ann_vol": vol * np.sqrt(WEEKS_PER_YEAR) if pd.notna(vol) else np.nan,
                "sharpe": sharpe,
                "avg_weekly_turnover": turnover.mean(),
                "observations": net_returns.dropna().shape[0],
            }
        )

    path_table = pd.DataFrame({"gross_return": gross_returns, "turnover": turnover}, index=score_panel.index)
    return pd.DataFrame(rows), path_table


def plot_decay_curve(summary_df, metric, title, ax=None, scale=1.0):
    ax = ax or plt.gca()
    if summary_df.empty or metric not in summary_df.columns:
        ax.set_title(f"{title}\n(no data)")
        return ax
    data = summary_df.sort_values("horizon_weeks")
    ax.plot(data["horizon_weeks"], data[metric] * scale, marker="o", lw=2)
    ax.axhline(0.0, color="black", lw=1, alpha=0.5)
    ax.set_xticks(list(FORWARD_HORIZONS))
    ax.set_title(title)
    ax.set_xlabel("Forward horizon (weeks)")
    return ax


cross_sectional_summaries = []
cross_sectional_ic_timeseries = []
time_series_summaries = []
cost_summaries = []
cross_signal_panels = {}
signal_manifest_records = []


def register_cross_signal(signal_name, score_panel):
    ic_summary, ic_ts = evaluate_cross_sectional_signal(signal_name, score_panel, forward_returns)
    cost_table, path_table = transaction_cost_table(signal_name, score_panel, one_week_ahead_returns)
    cross_sectional_summaries.append(ic_summary)
    if not ic_ts.empty:
        cross_sectional_ic_timeseries.append(ic_ts)
    cost_summaries.append(cost_table)
    cross_signal_panels[signal_name] = score_panel.copy()
    return ic_summary, ic_ts, cost_table, path_table


def register_time_series_signal(signal_name, signal_panel):
    ts_summary, bucket_table = evaluate_time_series_signal(signal_name, signal_panel, forward_returns)
    time_series_summaries.append(ts_summary)
    return ts_summary, bucket_table


def add_manifest_record(**kwargs):
    signal_manifest_records.append(kwargs)


## 3. Price Signal A1 — Time-Series Momentum (TSMOM)

**What the signal is**

Time-series momentum asks whether an ETF’s own past return predicts its own future return.
For weekly data we use the requested 52-week lookback and skip the most recent 4 weeks:

\[
\text{raw\_tsmom}_{t} = \frac{P_{t-4}}{P_{t-52}} - 1
\]

**Why it might work economically**

- medium-horizon trends can persist because macro shocks and institutional flows diffuse slowly
- ETFs bundle broad risk premia, so trend persistence can be cleaner than in single names
- skipping the latest 4 weeks reduces contamination from short-term reversal and mean reversion

**How it is implemented here**

- numerator: simple price return from week `t-52` to `t-4`
- denominator: annualized realized volatility from weekly log returns
- outputs: raw signal, realized vol, a cross-sectional score, and a lagged tradable version used in validation

**Timing note**

TSMOM is price-based, so it uses the common 1-week signal lag. The notebook exports both the observed score
and the tradable lagged score so downstream notebooks do not need to guess which one was validated.

**ETF caveats**

ETF momentum partly reflects underlying asset-class trends, not firm-specific information.
It is therefore economically closer to a small, diversified trend-following signal than to single-stock momentum.

**Output file**

- `data/02_layer1_signals/signal_tsmom.csv`


In [ ]:
tsmom_raw_observed = trailing_simple_return(weekly_prices, lookback=TSMOM_LOOKBACK, skip=TSMOM_SKIP)
tsmom_ann_vol = annualized_realized_vol(weekly_log_returns, window=TSMOM_VOL_WINDOW, min_periods=max(13, TSMOM_VOL_WINDOW // 2))
tsmom_vol_scaled_observed = tsmom_raw_observed.div(tsmom_ann_vol.replace(0, np.nan))
tsmom_score_observed = panel_score(tsmom_vol_scaled_observed, winsorize=True)
tsmom_bucket_observed = panel_bucket(tsmom_vol_scaled_observed)

tsmom_vol_scaled_tradable = apply_signal_lag(tsmom_vol_scaled_observed)
tsmom_score_tradable = apply_signal_lag(tsmom_score_observed)

tsmom_long = panel_dict_to_long(
    {
        "raw_tsmom_52_4w": tsmom_raw_observed,
        "realized_vol_ann_26w": tsmom_ann_vol,
        "tsmom_vol_scaled_observed": tsmom_vol_scaled_observed,
        "tsmom_score_observed": tsmom_score_observed,
        "tsmom_bucket_observed": tsmom_bucket_observed,
        "tsmom_vol_scaled_tradable": tsmom_vol_scaled_tradable,
        "tsmom_score_tradable": tsmom_score_tradable,
    }
).merge(asset_class_frame, on="Ticker", how="left")
tsmom_path = OUTPUT_DIR / "signal_tsmom.csv"
tsmom_long.to_csv(tsmom_path, index=False)

tsmom_ic_summary, _, tsmom_cost_table, _ = register_cross_signal("tsmom_vol_scaled", tsmom_score_tradable)
tsmom_ts_summary, _ = register_time_series_signal("tsmom_vol_scaled", tsmom_vol_scaled_tradable)

add_manifest_record(
    signal_name="tsmom_vol_scaled",
    file_name="signal_tsmom.csv",
    description="52-4 week own-price momentum scaled by 26-week realized volatility.",
    category="price_momentum",
    cross_sectional_or_time_series="both",
    required_inputs=["weekly_prices.csv", "weekly_returns.csv"],
    lookback="52 weeks",
    skip_period="4 weeks",
    lag_applied=total_lag_weeks("price"),
    normalization_method="winsorized cross-sectional rank to [-1, +1] for the cross-sectional score",
    caveats="Price-based ETF momentum still loads on broad market and asset-class trends.",
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_signal_examples(tsmom_vol_scaled_observed, "Observed vol-scaled TSMOM examples", ax=axes[0])
plot_decay_curve(tsmom_ic_summary, "mean_ic", "Tradable TSMOM IC decay", ax=axes[1])
plot_decay_curve(tsmom_ts_summary, "signed_strategy_mean", "Directional spread by horizon", ax=axes[2], scale=100)
axes[2].set_ylabel("Average signed forward return (%)")
plt.tight_layout()
plt.show()

print(f"Saved: {tsmom_path}")
print()
print("Cross-sectional validation")
print(tsmom_ic_summary.round(4).to_string(index=False))
print()
print("Time-series validation")
print(tsmom_ts_summary.round(4).to_string(index=False))
print()
print("Transaction-cost sensitivity")
print(tsmom_cost_table.round(4).to_string(index=False))


## 4. Price Signal A2 — Cross-Sectional Momentum (XSMOM)

**What the signal is**

Cross-sectional momentum ranks ETFs against each other at each date using the same 52-to-4 week lookback return.
Instead of asking “is SPY trending?”, it asks “which ETFs are stronger than the rest of the tradeable universe?”

**Why it might work economically**

- winners can keep outperforming losers because capital reallocations happen gradually
- relative strength is often more stable than absolute forecasts in diversified ETF universes
- ranking makes the signal robust to different volatility levels across asset classes

**How it is implemented here**

- global cross-sectional rank across the full ETF universe
- asset-class-neutral rank within `Equities`, `Bonds`, `Commodities`, `REITs`, and `FX` where group size permits
- normalized scores in `[-1, +1]`

**Why the neutral version is useful**

In ETF universes, global momentum can end up being dominated by broad asset-class moves.
The neutral version asks a stricter question: *within the same asset class, which ETFs are strongest?*

**ETF caveats**

Some asset classes in this universe are thin.
In particular, REITs and FX have only one main ETF each here, so neutral ranking for those groups is intentionally left missing.

**Output file**

- `data/02_layer1_signals/signal_xsmom.csv`


In [ ]:
xsmom_raw_observed = tsmom_raw_observed.copy()
xsmom_raw_rank = panel_rank(xsmom_raw_observed)
xsmom_score_observed = panel_score(xsmom_raw_observed, winsorize=True)
xsmom_score_tradable = apply_signal_lag(xsmom_score_observed)
xsmom_bucket_observed = panel_bucket(xsmom_raw_observed)

xsmom_asset_neutral_observed = grouped_panel_score(
    xsmom_raw_observed,
    asset_class_map,
    min_group_size=ASSET_CLASS_NEUTRAL_MIN_GROUP_SIZE,
)
xsmom_asset_neutral_tradable = apply_signal_lag(xsmom_asset_neutral_observed)

xsmom_long = panel_dict_to_long(
    {
        "xsmom_raw_return_52_4w": xsmom_raw_observed,
        "xsmom_raw_rank": xsmom_raw_rank,
        "xsmom_score_observed": xsmom_score_observed,
        "xsmom_score_tradable": xsmom_score_tradable,
        "xsmom_bucket_observed": xsmom_bucket_observed,
        "xsmom_asset_class_neutral_observed": xsmom_asset_neutral_observed,
        "xsmom_asset_class_neutral_tradable": xsmom_asset_neutral_tradable,
    }
).merge(asset_class_frame, on="Ticker", how="left")
xsmom_path = OUTPUT_DIR / "signal_xsmom.csv"
xsmom_long.to_csv(xsmom_path, index=False)

xsmom_ic_summary, _, xsmom_cost_table, _ = register_cross_signal("xsmom_global", xsmom_score_tradable)

add_manifest_record(
    signal_name="xsmom_global",
    file_name="signal_xsmom.csv",
    description="Global 52-4 week cross-sectional ETF momentum score.",
    category="price_momentum",
    cross_sectional_or_time_series="cross_sectional",
    required_inputs=["weekly_prices.csv"],
    lookback="52 weeks",
    skip_period="4 weeks",
    lag_applied=total_lag_weeks("price"),
    normalization_method="winsorized cross-sectional rank to [-1, +1]",
    caveats="Global ranking can be dominated by broad asset-class trends.",
)

if xsmom_asset_neutral_tradable.notna().sum().sum() > 0:
    xsmom_asset_ic_summary, _, xsmom_asset_cost_table, _ = register_cross_signal(
        "xsmom_asset_class_neutral",
        xsmom_asset_neutral_tradable,
    )
    add_manifest_record(
        signal_name="xsmom_asset_class_neutral",
        file_name="signal_xsmom.csv",
        description="52-4 week ETF momentum ranked within asset class when group size permits.",
        category="price_momentum",
        cross_sectional_or_time_series="cross_sectional",
        required_inputs=["weekly_prices.csv", "universe_metadata.csv"],
        lookback="52 weeks",
        skip_period="4 weeks",
        lag_applied=total_lag_weeks("price"),
        normalization_method="within-asset-class rank to [-1, +1]",
        caveats="Groups with only one ETF are left missing by design.",
    )
else:
    xsmom_asset_ic_summary = pd.DataFrame()
    xsmom_asset_cost_table = pd.DataFrame()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_signal_examples(xsmom_score_observed, "Observed XSMOM scores", ax=axes[0])
plot_decay_curve(xsmom_ic_summary, "mean_ic", "Global XSMOM IC decay", ax=axes[1])
plot_decay_curve(xsmom_asset_ic_summary, "mean_ic", "Asset-neutral XSMOM IC decay", ax=axes[2])
plt.tight_layout()
plt.show()

print(f"Saved: {xsmom_path}")
print()
print("Global XSMOM")
print(xsmom_ic_summary.round(4).to_string(index=False))
print(xsmom_cost_table.round(4).to_string(index=False))
print()
if not xsmom_asset_ic_summary.empty:
    print("Asset-class-neutral XSMOM")
    print(xsmom_asset_ic_summary.round(4).to_string(index=False))
    print(xsmom_asset_cost_table.round(4).to_string(index=False))
else:
    print("Asset-class-neutral XSMOM was skipped because no asset class had enough members.")


## 5. Price Signal A3 — Short-Horizon Reversal

**What the signal is**

Reversal says that recent losers can bounce and recent winners can mean-revert over short horizons.
We test two ETF variants:

- 1-week reversal
- 4-week reversal

**Why it might work economically**

- short-horizon moves can overshoot because of positioning, flows, and volatility shocks
- ETF baskets can mean-revert after fast risk-on / risk-off bursts
- the effect usually decays quickly, so we expect the shortest forward horizons to matter most

**How it is implemented here**

- raw recent return = 1-week or 4-week simple return
- reversal score = negative cross-sectional rank of that recent return
- both global and asset-class-neutral versions are tested when asset-class labels are available

**ETF caveats**

Reversal is usually more convincing cross-sectionally than time-series-wise. In ETFs it can also be regime-dependent:
crisis periods often strengthen reversal, while clean trend periods weaken it.

**Output file**

- `data/02_layer1_signals/signal_reversal.csv`


In [ ]:
reversal_1w_raw_observed = weekly_simple_returns.copy()
reversal_4w_raw_observed = trailing_simple_return(weekly_prices, lookback=4, skip=0)

reversal_1w_score_observed = panel_score(-reversal_1w_raw_observed, winsorize=True)
reversal_4w_score_observed = panel_score(-reversal_4w_raw_observed, winsorize=True)
reversal_1w_tradable = apply_signal_lag(reversal_1w_score_observed)
reversal_4w_tradable = apply_signal_lag(reversal_4w_score_observed)

reversal_1w_asset_neutral_observed = grouped_panel_score(
    -reversal_1w_raw_observed,
    asset_class_map,
    min_group_size=ASSET_CLASS_NEUTRAL_MIN_GROUP_SIZE,
)
reversal_4w_asset_neutral_observed = grouped_panel_score(
    -reversal_4w_raw_observed,
    asset_class_map,
    min_group_size=ASSET_CLASS_NEUTRAL_MIN_GROUP_SIZE,
)
reversal_1w_asset_neutral_tradable = apply_signal_lag(reversal_1w_asset_neutral_observed)
reversal_4w_asset_neutral_tradable = apply_signal_lag(reversal_4w_asset_neutral_observed)

reversal_long = panel_dict_to_long(
    {
        "recent_return_1w": reversal_1w_raw_observed,
        "reversal_1w_score_observed": reversal_1w_score_observed,
        "reversal_1w_score_tradable": reversal_1w_tradable,
        "reversal_1w_asset_class_neutral_observed": reversal_1w_asset_neutral_observed,
        "reversal_1w_asset_class_neutral_tradable": reversal_1w_asset_neutral_tradable,
        "recent_return_4w": reversal_4w_raw_observed,
        "reversal_4w_score_observed": reversal_4w_score_observed,
        "reversal_4w_score_tradable": reversal_4w_tradable,
        "reversal_4w_asset_class_neutral_observed": reversal_4w_asset_neutral_observed,
        "reversal_4w_asset_class_neutral_tradable": reversal_4w_asset_neutral_tradable,
    }
).merge(asset_class_frame, on="Ticker", how="left")
reversal_path = OUTPUT_DIR / "signal_reversal.csv"
reversal_long.to_csv(reversal_path, index=False)

reversal_1w_ic_summary, _, reversal_1w_cost_table, _ = register_cross_signal("reversal_1w_global", reversal_1w_tradable)
reversal_4w_ic_summary, _, reversal_4w_cost_table, _ = register_cross_signal("reversal_4w_global", reversal_4w_tradable)

add_manifest_record(
    signal_name="reversal_1w_global",
    file_name="signal_reversal.csv",
    description="1-week ETF reversal ranked across the whole universe.",
    category="price_reversal",
    cross_sectional_or_time_series="cross_sectional",
    required_inputs=["weekly_prices.csv"],
    lookback="1 week",
    skip_period="0 weeks",
    lag_applied=total_lag_weeks("price"),
    normalization_method="winsorized cross-sectional rank to [-1, +1]",
    caveats="Fast, noisy, and highly regime-dependent.",
)
add_manifest_record(
    signal_name="reversal_4w_global",
    file_name="signal_reversal.csv",
    description="4-week ETF reversal ranked across the whole universe.",
    category="price_reversal",
    cross_sectional_or_time_series="cross_sectional",
    required_inputs=["weekly_prices.csv"],
    lookback="4 weeks",
    skip_period="0 weeks",
    lag_applied=total_lag_weeks("price"),
    normalization_method="winsorized cross-sectional rank to [-1, +1]",
    caveats="Slower than 1-week reversal and often weaker in clean trend regimes.",
)

if reversal_1w_asset_neutral_tradable.notna().sum().sum() > 0:
    reversal_1w_asset_ic_summary, _, reversal_1w_asset_cost_table, _ = register_cross_signal(
        "reversal_1w_asset_class_neutral",
        reversal_1w_asset_neutral_tradable,
    )
    add_manifest_record(
        signal_name="reversal_1w_asset_class_neutral",
        file_name="signal_reversal.csv",
        description="1-week ETF reversal ranked within asset class.",
        category="price_reversal",
        cross_sectional_or_time_series="cross_sectional",
        required_inputs=["weekly_prices.csv", "universe_metadata.csv"],
        lookback="1 week",
        skip_period="0 weeks",
        lag_applied=total_lag_weeks("price"),
        normalization_method="within-asset-class rank to [-1, +1]",
        caveats="Groups with too few members are intentionally left missing.",
    )
else:
    reversal_1w_asset_ic_summary = pd.DataFrame()
    reversal_1w_asset_cost_table = pd.DataFrame()

if reversal_4w_asset_neutral_tradable.notna().sum().sum() > 0:
    reversal_4w_asset_ic_summary, _, reversal_4w_asset_cost_table, _ = register_cross_signal(
        "reversal_4w_asset_class_neutral",
        reversal_4w_asset_neutral_tradable,
    )
    add_manifest_record(
        signal_name="reversal_4w_asset_class_neutral",
        file_name="signal_reversal.csv",
        description="4-week ETF reversal ranked within asset class.",
        category="price_reversal",
        cross_sectional_or_time_series="cross_sectional",
        required_inputs=["weekly_prices.csv", "universe_metadata.csv"],
        lookback="4 weeks",
        skip_period="0 weeks",
        lag_applied=total_lag_weeks("price"),
        normalization_method="within-asset-class rank to [-1, +1]",
        caveats="Groups with too few members are intentionally left missing.",
    )
else:
    reversal_4w_asset_ic_summary = pd.DataFrame()
    reversal_4w_asset_cost_table = pd.DataFrame()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_decay_curve(reversal_1w_ic_summary, "mean_ic", "1-week global reversal IC decay", ax=axes[0, 0])
plot_decay_curve(reversal_4w_ic_summary, "mean_ic", "4-week global reversal IC decay", ax=axes[0, 1])
plot_decay_curve(reversal_1w_asset_ic_summary, "mean_ic", "1-week asset-neutral reversal IC decay", ax=axes[1, 0])
plot_decay_curve(reversal_4w_asset_ic_summary, "mean_ic", "4-week asset-neutral reversal IC decay", ax=axes[1, 1])
plt.tight_layout()
plt.show()

print(f"Saved: {reversal_path}")
print()
print("1-week global reversal")
print(reversal_1w_ic_summary.round(4).to_string(index=False))
print(reversal_1w_cost_table.round(4).to_string(index=False))
print()
print("4-week global reversal")
print(reversal_4w_ic_summary.round(4).to_string(index=False))
print(reversal_4w_cost_table.round(4).to_string(index=False))
print()
if not reversal_1w_asset_ic_summary.empty:
    print("1-week asset-class-neutral reversal")
    print(reversal_1w_asset_ic_summary.round(4).to_string(index=False))
    print(reversal_1w_asset_cost_table.round(4).to_string(index=False))
    print()
if not reversal_4w_asset_ic_summary.empty:
    print("4-week asset-class-neutral reversal")
    print(reversal_4w_asset_ic_summary.round(4).to_string(index=False))
    print(reversal_4w_asset_cost_table.round(4).to_string(index=False))


## 6. Price Signals A4 and A5 — Multi-Horizon Momentum Blend and Residual Momentum

### A4. Multi-Horizon TSMOM Blend

**What the signal is**

Instead of relying on one lookback window, we blend momentum measured at 13, 26, 39, and 52 weeks.
The idea is to keep the signal from being overly dependent on one arbitrary horizon.

**Why it might work economically**

- different trend horizons can capture different types of persistence
- shorter windows react faster, longer windows are often more stable
- blended signals are usually less brittle than a single raw window

**How it is implemented here**

For each horizon `h`, we compute:

\[
\frac{P_{t-4}}{P_{t-h}} - 1
\]

We then test two blends:

- equal-weight blend
- inverse signal-volatility blend, where more unstable component signals get less weight

### A5. Residual Momentum

**What the signal is**

Residual momentum removes broad market beta before measuring momentum.
Here we:

- estimate a rolling beta model of each ETF versus the explicit market proxy
- compute residual log returns after removing market exposure
- build momentum on those residual returns over the same 52-to-4 style formation window

**Why this helps**

ETF universes often move together. A raw momentum signal can therefore reward broad beta more than true relative strength.
Residual momentum asks a sharper question: *after stripping out the common market move, which ETFs still look strong?*

**ETF caveats**

Residual momentum is still not fully idiosyncratic in an ETF universe because sector, duration, credit, commodity, and dollar betas remain.
But it is usually cleaner than plain market-loaded momentum when many assets are co-moving.

**Output files**

- `data/02_layer1_signals/signal_multi_horizon_mom.csv`
- `data/02_layer1_signals/signal_residual_momentum.csv`


In [ ]:
multi_components = {
    "mom_13_4w": trailing_simple_return(weekly_prices, lookback=13, skip=4),
    "mom_26_4w": trailing_simple_return(weekly_prices, lookback=26, skip=4),
    "mom_39_4w": trailing_simple_return(weekly_prices, lookback=39, skip=4),
    "mom_52_4w": trailing_simple_return(weekly_prices, lookback=52, skip=4),
}

multi_equal_observed = sum(multi_components.values()) / len(multi_components)
component_vols = {
    name: component.rolling(MULTI_BLEND_VOL_WINDOW, min_periods=max(13, MULTI_BLEND_VOL_WINDOW // 2)).std()
    for name, component in multi_components.items()
}
# Practical floor: a near-zero component volatility can create meaningless inverse-vol weights.
inverse_vol_weights = {
    name: 1.0 / vol.clip(lower=MULTI_BLEND_MIN_COMPONENT_VOL)
    for name, vol in component_vols.items()
}
inverse_weight_sum = sum(inverse_vol_weights.values())
multi_invvol_observed = sum(inverse_vol_weights[name] * multi_components[name] for name in multi_components) / inverse_weight_sum

multi_equal_score_observed = panel_score(multi_equal_observed, winsorize=True)
multi_invvol_score_observed = panel_score(multi_invvol_observed, winsorize=True)
multi_equal_tradable = apply_signal_lag(multi_equal_score_observed)
multi_invvol_tradable = apply_signal_lag(multi_invvol_score_observed)

multi_mom_long = panel_dict_to_long(
    {
        **multi_components,
        "multi_mom_equal_observed": multi_equal_observed,
        "multi_mom_equal_score_observed": multi_equal_score_observed,
        "multi_mom_equal_score_tradable": multi_equal_tradable,
        "multi_mom_invvol_observed": multi_invvol_observed,
        "multi_mom_invvol_score_observed": multi_invvol_score_observed,
        "multi_mom_invvol_score_tradable": multi_invvol_tradable,
    }
).merge(asset_class_frame, on="Ticker", how="left")
multi_mom_path = OUTPUT_DIR / "signal_multi_horizon_mom.csv"
multi_mom_long.to_csv(multi_mom_path, index=False)

multi_equal_ic_summary, _, multi_equal_cost_table, _ = register_cross_signal("multi_mom_equal", multi_equal_tradable)
multi_invvol_ic_summary, _, multi_invvol_cost_table, _ = register_cross_signal("multi_mom_invvol", multi_invvol_tradable)
multi_equal_ts_summary, _ = register_time_series_signal("multi_mom_equal", apply_signal_lag(multi_equal_observed))
multi_invvol_ts_summary, _ = register_time_series_signal("multi_mom_invvol", apply_signal_lag(multi_invvol_observed))

add_manifest_record(
    signal_name="multi_mom_equal",
    file_name="signal_multi_horizon_mom.csv",
    description="Equal-weight blend of 13, 26, 39, and 52 week 4-skip momentum windows.",
    category="price_momentum",
    cross_sectional_or_time_series="both",
    required_inputs=["weekly_prices.csv"],
    lookback="13/26/39/52 weeks",
    skip_period="4 weeks",
    lag_applied=total_lag_weeks("price"),
    normalization_method="winsorized cross-sectional rank to [-1, +1]",
    caveats="Blended trend signal; still exposed to broad asset-class momentum.",
)
add_manifest_record(
    signal_name="multi_mom_invvol",
    file_name="signal_multi_horizon_mom.csv",
    description="Inverse-volatility weighted blend of 13, 26, 39, and 52 week 4-skip momentum windows.",
    category="price_momentum",
    cross_sectional_or_time_series="both",
    required_inputs=["weekly_prices.csv"],
    lookback="13/26/39/52 weeks",
    skip_period="4 weeks",
    lag_applied=total_lag_weeks("price"),
    normalization_method="winsorized cross-sectional rank to [-1, +1]",
    caveats="More stable than a single window, but still trend-following and not inherently orthogonal to XSMOM. Inverse-vol component weights use a practical 1% volatility floor to avoid unstable weight spikes.",
)

residual_log_returns, residual_beta_used, residual_alpha_used = compute_residual_log_returns(
    weekly_log_returns,
    market_proxy_log_return,
    window=RESIDUAL_BETA_WINDOW,
    min_periods=BETA_MIN_PERIODS,
)
residual_mom_log_observed = residual_log_returns.shift(TSMOM_SKIP).rolling(
    RESIDUAL_MOM_FORMATION_WEEKS,
    min_periods=max(20, RESIDUAL_MOM_FORMATION_WEEKS // 2),
).sum()
residual_mom_observed = np.expm1(residual_mom_log_observed)
residual_mom_score_observed = panel_score(residual_mom_observed, winsorize=True)
residual_mom_score_tradable = apply_signal_lag(residual_mom_score_observed)
residual_mom_ts_tradable = apply_signal_lag(residual_mom_observed)

residual_mom_long = panel_dict_to_long(
    {
        "residual_beta_used": residual_beta_used,
        "residual_alpha_used": residual_alpha_used,
        "residual_log_return": residual_log_returns,
        "residual_mom_52_4w_observed": residual_mom_observed,
        "residual_mom_score_observed": residual_mom_score_observed,
        "residual_mom_score_tradable": residual_mom_score_tradable,
    }
).merge(asset_class_frame, on="Ticker", how="left")
residual_mom_path = OUTPUT_DIR / "signal_residual_momentum.csv"
residual_mom_long.to_csv(residual_mom_path, index=False)

residual_ic_summary, _, residual_cost_table, _ = register_cross_signal("residual_momentum", residual_mom_score_tradable)
residual_ts_summary, _ = register_time_series_signal("residual_momentum", residual_mom_ts_tradable)

add_manifest_record(
    signal_name="residual_momentum",
    file_name="signal_residual_momentum.csv",
    description="52-4 week momentum on residual log returns after removing rolling market beta.",
    category="price_momentum",
    cross_sectional_or_time_series="both",
    required_inputs=["weekly_returns.csv", "market_proxy_weekly.csv", "proxy_mapping.json"],
    lookback="52 weeks",
    skip_period="4 weeks",
    lag_applied=total_lag_weeks("price"),
    normalization_method="winsorized cross-sectional rank to [-1, +1]",
    caveats=f"Residualized against {market_proxy_ticker}; still not free of sector, duration, or commodity betas.",
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_signal_examples(multi_equal_observed, "Observed equal-weight multi-horizon momentum", ax=axes[0])
plot_decay_curve(multi_equal_ic_summary, "mean_ic", "Equal-weight multi-horizon IC decay", ax=axes[1])
plot_decay_curve(residual_ic_summary, "mean_ic", "Residual momentum IC decay", ax=axes[2])
plt.tight_layout()
plt.show()

print(f"Saved: {multi_mom_path}")
print(f"Saved: {residual_mom_path}")
print()
print("Equal-weight blend")
print(multi_equal_ic_summary.round(4).to_string(index=False))
print(multi_equal_ts_summary.round(4).to_string(index=False))
print(multi_equal_cost_table.round(4).to_string(index=False))
print()
print("Inverse-volatility blend")
print(multi_invvol_ic_summary.round(4).to_string(index=False))
print(multi_invvol_ts_summary.round(4).to_string(index=False))
print(multi_invvol_cost_table.round(4).to_string(index=False))
print()
print("Residual momentum")
print(residual_ic_summary.round(4).to_string(index=False))
print(residual_ts_summary.round(4).to_string(index=False))
print(residual_cost_table.round(4).to_string(index=False))


## 7. Proxy Factor Signal B5 — Carry for ETFs

**What the signal is**

In futures or FX, carry has a very specific meaning. In ETFs, we usually do **not** have a clean asset-level carry series.
The practical proxy here is trailing distribution yield:

\[
\text{carry proxy}_{t} \approx \frac{\text{distributions over last 52 weeks}}{P_t}
\]

**What is implemented vs what is not**

- **Implemented**: a historical ETF carry proxy using cached distribution history when available
- **Not implemented**: true futures roll yield, FX carry, financing carry, or constituent-level carry decomposition

**Where this proxy is strongest vs weakest**

- **Bonds / REITs**: strongest practical use because distributions are more central to the economics
- **Equity ETFs**: usable as an income / distribution-yield proxy, but not the same as true carry
- **Commodities**: weak because ETF distributions often miss roll yield
- **FX**: weak to unavailable in this ETF set

**Implementation caveat**

This proxy uses **trailing distributions divided by the current ETF price**. That is a practical approximation, not a payment-date-based yield calculation. The level therefore should be read as a comparative cross-sectional proxy more than a precise economic carry measure.

**Timing note**

Distribution history and Yahoo yield metadata are treated as external inputs, so the tradable carry signal is lagged more conservatively than price-only signals.

**Output file**

- `data/02_layer1_signals/signal_carry.csv`


In [ ]:
carry_yield_observed, carry_meta = build_carry_proxy(
    research_universe,
    weekly_prices,
    distribution_history,
    yahoo_metadata_snapshot,
    trailing_weeks=CARRY_TRAILING_WEEKS,
    live_fallback=distribution_history.empty,
)
carry_meta = normalize_ticker_schema(carry_meta, preferred="ticker")
carry_meta["asset_class"] = carry_meta["ticker"].map(asset_class_map)

carry_score_observed = panel_score(carry_yield_observed, winsorize=True)
carry_score_tradable = apply_signal_lag(carry_score_observed, extra_lag_weeks=EXTERNAL_DATA_LAG_WEEKS)
carry_yield_tradable = apply_signal_lag(carry_yield_observed, extra_lag_weeks=EXTERNAL_DATA_LAG_WEEKS)
carry_asset_neutral_observed = grouped_panel_score(
    carry_yield_observed,
    asset_class_map,
    min_group_size=ASSET_CLASS_NEUTRAL_MIN_GROUP_SIZE,
)
carry_asset_neutral_tradable = apply_signal_lag(
    carry_asset_neutral_observed,
    extra_lag_weeks=EXTERNAL_DATA_LAG_WEEKS,
)

carry_long = panel_dict_to_long(
    {
        "carry_yield_trailing_52w_observed": carry_yield_observed,
        "carry_yield_trailing_52w_tradable": carry_yield_tradable,
        "carry_score_observed": carry_score_observed,
        "carry_score_tradable": carry_score_tradable,
        "carry_score_asset_class_neutral_observed": carry_asset_neutral_observed,
        "carry_score_asset_class_neutral_tradable": carry_asset_neutral_tradable,
    }
).merge(normalize_ticker_schema(carry_meta, preferred="Ticker"), on="Ticker", how="left")
carry_path = OUTPUT_DIR / "signal_carry.csv"
carry_long.to_csv(carry_path, index=False)

carry_ic_summary, _, carry_cost_table, _ = register_cross_signal("carry_proxy", carry_score_tradable)

add_manifest_record(
    signal_name="carry_proxy",
    file_name="signal_carry.csv",
    description="Trailing 52-week ETF distribution-yield proxy ranked cross-sectionally.",
    category="carry_proxy",
    cross_sectional_or_time_series="cross_sectional",
    required_inputs=["weekly_prices.csv", "etf_distribution_history.csv", "yahoo_metadata_snapshot.csv"],
    lookback="52 weeks",
    skip_period="0 weeks",
    lag_applied=total_lag_weeks("external"),
    normalization_method="winsorized cross-sectional rank to [-1, +1]",
    caveats="A proxy only; strongest for bonds and REITs, weakest for commodities and FX. Trailing distributions are divided by current price, so the level is approximate and best used cross-sectionally.",
)

if carry_asset_neutral_tradable.notna().sum().sum() > 0:
    carry_asset_ic_summary, _, carry_asset_cost_table, _ = register_cross_signal(
        "carry_proxy_asset_class_neutral",
        carry_asset_neutral_tradable,
    )
    add_manifest_record(
        signal_name="carry_proxy_asset_class_neutral",
        file_name="signal_carry.csv",
        description="Trailing 52-week ETF carry proxy ranked within asset class when enough peers exist.",
        category="carry_proxy",
        cross_sectional_or_time_series="cross_sectional",
        required_inputs=["weekly_prices.csv", "etf_distribution_history.csv", "yahoo_metadata_snapshot.csv", "universe_metadata.csv"],
        lookback="52 weeks",
        skip_period="0 weeks",
        lag_applied=total_lag_weeks("external"),
        normalization_method="within-asset-class rank to [-1, +1]",
        caveats="Most useful in bonds and REITs; skipped for thin asset classes. Trailing distributions divided by current price are still an approximation, so interpretation should stay comparative.",
    )
else:
    carry_asset_ic_summary = pd.DataFrame()
    carry_asset_cost_table = pd.DataFrame()

carry_coverage_summary = coverage_by_asset_class(carry_yield_observed, asset_class_map)
# Bug fix note: cached and live carry inputs were not guaranteed to expose the ticker field with the same
# casing or as an actual column. We now normalize ticker-like schemas before the carry merge.
carry_source_summary = (
    carry_meta.groupby(["asset_class", "carry_source"], dropna=False)
    .size()
    .reset_index(name="tickers")
    .sort_values(["asset_class", "tickers"], ascending=[True, False])
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_signal_examples(carry_yield_observed, "Observed trailing distribution yield", ax=axes[0])
plot_decay_curve(carry_ic_summary, "mean_ic", "Carry proxy IC decay", ax=axes[1])
plot_decay_curve(carry_asset_ic_summary, "mean_ic", "Carry proxy asset-neutral IC decay", ax=axes[2])
plt.tight_layout()
plt.show()

print(f"Saved: {carry_path}")
print()
print("Carry source audit")
print(carry_meta["carry_source"].value_counts(dropna=False).to_string())
print()
print("Coverage by asset class")
print(carry_coverage_summary.round(2).to_string(index=False))
print()
print("Source detail by asset class")
print(carry_source_summary.to_string(index=False))
print()
print("Global carry proxy")
print(carry_ic_summary.round(4).to_string(index=False))
print(carry_cost_table.round(4).to_string(index=False))
print()
if not carry_asset_ic_summary.empty:
    print("Asset-class-neutral carry proxy")
    print(carry_asset_ic_summary.round(4).to_string(index=False))
    print(carry_asset_cost_table.round(4).to_string(index=False))
else:
    print("Asset-class-neutral carry proxy was skipped because no asset class had enough members.")


## 8. Proxy Factor Signal B6 — Value for ETFs

**What the signal is**

Because ETF accounting fundamentals are not consistently available in the upstream data hub, we use an ETF-friendly proxy:

\[
\text{value gap}_{t} = \frac{\text{5-year moving average price}}{P_t} - 1
\]

We also test a long-horizon z-score of price relative to its own moving average.

**Why it might work economically**

- large deviations below long-run price anchors can capture mean reversion or normalization
- for diversified ETFs, this can proxy for “cheap versus own history,” especially in rates and sector rotations

**How it is implemented here**

- 260-week moving average as a 5-year anchor
- value gap and price-vs-MA z-score are standardized cross-sectionally and blended
- the tradable signal follows the price-feature lag convention

**ETF caveats**

This is **not** book-to-price or earnings yield. It is a “relative-to-own-history value” measure, which is closer to slow mean reversion than to classic equity value.

**Output file**

- `data/02_layer1_signals/signal_value.csv`


In [ ]:
value_ma_5y = weekly_prices.rolling(VALUE_WINDOW, min_periods=VALUE_MIN_PERIODS).mean()
value_gap_observed = value_ma_5y.div(weekly_prices) - 1.0
value_price_z_observed = (weekly_prices - value_ma_5y).div(
    weekly_prices.rolling(VALUE_WINDOW, min_periods=VALUE_MIN_PERIODS).std() + 1e-12
)

value_gap_score_observed = panel_score(value_gap_observed, winsorize=True)
value_z_score_observed = panel_score(-value_price_z_observed, winsorize=True)
value_composite_observed = (value_gap_score_observed + value_z_score_observed) / 2.0
value_composite_tradable = apply_signal_lag(value_composite_observed)
value_asset_neutral_observed = grouped_panel_score(
    value_composite_observed,
    asset_class_map,
    min_group_size=ASSET_CLASS_NEUTRAL_MIN_GROUP_SIZE,
)
value_asset_neutral_tradable = apply_signal_lag(value_asset_neutral_observed)

value_long = panel_dict_to_long(
    {
        "value_gap_5y_ma_over_price": value_gap_observed,
        "value_price_vs_ma_z_observed": value_price_z_observed,
        "value_gap_score_observed": value_gap_score_observed,
        "value_z_score_observed": value_z_score_observed,
        "value_score_observed": value_composite_observed,
        "value_score_tradable": value_composite_tradable,
        "value_score_asset_class_neutral_observed": value_asset_neutral_observed,
        "value_score_asset_class_neutral_tradable": value_asset_neutral_tradable,
    }
).merge(asset_class_frame, on="Ticker", how="left")
value_path = OUTPUT_DIR / "signal_value.csv"
value_long.to_csv(value_path, index=False)

value_ic_summary, _, value_cost_table, _ = register_cross_signal("value_proxy", value_composite_tradable)

add_manifest_record(
    signal_name="value_proxy",
    file_name="signal_value.csv",
    description="ETF value proxy based on 5-year moving-average gap and long-horizon price z-score.",
    category="value_proxy",
    cross_sectional_or_time_series="cross_sectional",
    required_inputs=["weekly_prices.csv"],
    lookback="260 weeks",
    skip_period="0 weeks",
    lag_applied=total_lag_weeks("price"),
    normalization_method="cross-sectional rank blend of value-gap and price-vs-MA z-score",
    caveats="Not true fundamental value; best interpreted as slow mean reversion to own history.",
)

if value_asset_neutral_tradable.notna().sum().sum() > 0:
    value_asset_ic_summary, _, value_asset_cost_table, _ = register_cross_signal(
        "value_proxy_asset_class_neutral",
        value_asset_neutral_tradable,
    )
    add_manifest_record(
        signal_name="value_proxy_asset_class_neutral",
        file_name="signal_value.csv",
        description="ETF value proxy ranked within asset class using the composite own-history value score.",
        category="value_proxy",
        cross_sectional_or_time_series="cross_sectional",
        required_inputs=["weekly_prices.csv", "universe_metadata.csv"],
        lookback="260 weeks",
        skip_period="0 weeks",
        lag_applied=total_lag_weeks("price"),
        normalization_method="within-asset-class rank to [-1, +1]",
        caveats="Useful when global value is dominated by cross-asset level effects rather than within-bucket mispricing.",
    )
else:
    value_asset_ic_summary = pd.DataFrame()
    value_asset_cost_table = pd.DataFrame()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_signal_examples(value_composite_observed, "Observed value proxy scores", ax=axes[0])
plot_decay_curve(value_ic_summary, "mean_ic", "Value proxy IC decay", ax=axes[1])
plot_decay_curve(value_asset_ic_summary, "mean_ic", "Value proxy asset-neutral IC decay", ax=axes[2])
plt.tight_layout()
plt.show()

print(f"Saved: {value_path}")
print()
print("Global value proxy")
print(value_ic_summary.round(4).to_string(index=False))
print(value_cost_table.round(4).to_string(index=False))
print()
if not value_asset_ic_summary.empty:
    print("Asset-class-neutral value proxy")
    print(value_asset_ic_summary.round(4).to_string(index=False))
    print(value_asset_cost_table.round(4).to_string(index=False))
else:
    print("Asset-class-neutral value proxy was skipped because no asset class had enough members.")


## 9. Proxy Factor Signal B7 — Betting Against Beta (BAB) Proxy

**What the signal is**

The idea behind BAB is that low-beta assets have historically delivered stronger risk-adjusted returns than a simple CAPM view would predict.
Here we build a transparent ETF-level proxy:

- estimate rolling beta of each ETF to an explicit market proxy
- rank betas cross-sectionally
- prefer **lower** beta ETFs

**How it is implemented here**

- if daily returns are available upstream, beta is estimated on daily data and sampled weekly
- otherwise the notebook falls back to a weekly covariance / variance estimate
- the market proxy comes from the explicit benchmark / proxy files when available, rather than assuming `SPY` implicitly

**ETF caveats**

This is not the full Frazzini-Pedersen BAB construction. There is no explicit leverage financing leg, no beta-neutral portfolio optimization, and the market-proxy choice matters.

**Output file**

- `data/02_layer1_signals/signal_bab.csv`


In [ ]:
market_proxy_simple_return = np.expm1(market_proxy_log_return)

if not daily_log_returns.empty and market_proxy_ticker in daily_log_returns.columns:
    daily_simple_returns = np.expm1(daily_log_returns.reindex(columns=research_universe))
    daily_market_proxy = np.expm1(daily_log_returns[market_proxy_ticker])
    bab_beta_daily = rolling_beta_panel(daily_simple_returns, daily_market_proxy, window=126, min_periods=63)
    bab_beta_observed = bab_beta_daily.resample("W-FRI").last().reindex(weekly_prices.index)
    beta_estimation_frequency = "daily_to_weekly"
else:
    bab_beta_observed = rolling_beta_panel(
        weekly_simple_returns,
        market_proxy_simple_return,
        window=BETA_WINDOW,
        min_periods=BETA_MIN_PERIODS,
    )
    beta_estimation_frequency = "weekly_only"

bab_score_observed = panel_score(-bab_beta_observed, winsorize=True)
bab_score_tradable = apply_signal_lag(bab_score_observed)
bab_asset_neutral_observed = grouped_panel_score(
    -bab_beta_observed,
    asset_class_map,
    min_group_size=ASSET_CLASS_NEUTRAL_MIN_GROUP_SIZE,
)
bab_asset_neutral_tradable = apply_signal_lag(bab_asset_neutral_observed)

bab_long = panel_dict_to_long(
    {
        "beta_to_market_observed": bab_beta_observed,
        "bab_score_observed": bab_score_observed,
        "bab_score_tradable": bab_score_tradable,
        "bab_score_asset_class_neutral_observed": bab_asset_neutral_observed,
        "bab_score_asset_class_neutral_tradable": bab_asset_neutral_tradable,
    }
).merge(asset_class_frame, on="Ticker", how="left")
bab_long["market_proxy_ticker"] = market_proxy_ticker
bab_long["beta_estimation_frequency"] = beta_estimation_frequency
bab_path = OUTPUT_DIR / "signal_bab.csv"
bab_long.to_csv(bab_path, index=False)

bab_ic_summary, _, bab_cost_table, _ = register_cross_signal("bab_proxy", bab_score_tradable)

add_manifest_record(
    signal_name="bab_proxy",
    file_name="signal_bab.csv",
    description="ETF-level low-beta proxy using an explicit market benchmark.",
    category="low_beta_proxy",
    cross_sectional_or_time_series="cross_sectional",
    required_inputs=["weekly_returns.csv", "market_proxy_weekly.csv", "proxy_mapping.json"],
    lookback="52 weeks (or ~126 trading days when daily returns are available)",
    skip_period="0 weeks",
    lag_applied=total_lag_weeks("price"),
    normalization_method="winsorized negative-beta rank to [-1, +1]",
    caveats=f"Market proxy = {market_proxy_ticker}; this is a low-beta proxy, not the full institutional BAB construction.",
)

if bab_asset_neutral_tradable.notna().sum().sum() > 0:
    bab_asset_ic_summary, _, bab_asset_cost_table, _ = register_cross_signal(
        "bab_proxy_asset_class_neutral",
        bab_asset_neutral_tradable,
    )
    add_manifest_record(
        signal_name="bab_proxy_asset_class_neutral",
        file_name="signal_bab.csv",
        description="ETF-level low-beta proxy ranked within asset class rather than globally.",
        category="low_beta_proxy",
        cross_sectional_or_time_series="cross_sectional",
        required_inputs=["weekly_returns.csv", "market_proxy_weekly.csv", "proxy_mapping.json", "universe_metadata.csv"],
        lookback="52 weeks (or ~126 trading days when daily returns are available)",
        skip_period="0 weeks",
        lag_applied=total_lag_weeks("price"),
        normalization_method="within-asset-class rank to [-1, +1]",
        caveats=f"Market proxy = {market_proxy_ticker}; mainly useful where enough same-asset-class peers exist.",
    )
else:
    bab_asset_ic_summary = pd.DataFrame()
    bab_asset_cost_table = pd.DataFrame()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_signal_examples(bab_beta_observed, f"Observed beta to {market_proxy_ticker}", ax=axes[0])
plot_decay_curve(bab_ic_summary, "mean_ic", "BAB proxy IC decay", ax=axes[1])
plot_decay_curve(bab_asset_ic_summary, "mean_ic", "BAB proxy asset-neutral IC decay", ax=axes[2])
plt.tight_layout()
plt.show()

print(f"Saved: {bab_path}")
print()
print(f"Beta estimation frequency: {beta_estimation_frequency}")
print("Global BAB proxy")
print(bab_ic_summary.round(4).to_string(index=False))
print(bab_cost_table.round(4).to_string(index=False))
print()
if not bab_asset_ic_summary.empty:
    print("Asset-class-neutral BAB proxy")
    print(bab_asset_ic_summary.round(4).to_string(index=False))
    print(bab_asset_cost_table.round(4).to_string(index=False))
else:
    print("Asset-class-neutral BAB proxy was skipped because no asset class had enough members.")


## 10. Proxy Factor Signal B8 — Quality for ETFs

**What the signal is**

ETF-level “quality” is less about corporate balance sheets and more about the stability of the return path.
We build a transparent composite using:

- inverse realized volatility
- inverse downside semivolatility
- inverse drawdown frequency
- inverse drawdown severity

**Why it might work economically**

- stable ETFs often sit in higher-quality sectors, safer duration buckets, or persistent defensive trends
- lower downside risk can proxy for “quality of the return path,” even when issuer-level accounting data are unavailable

**How it is implemented here**

Each raw stability metric is converted into a cross-sectional score and averaged.

**ETF caveats**

This is an intentionally modest proxy. It captures stability, not business quality in the classic stock-factor sense.

**Output file**

- `data/02_layer1_signals/signal_quality.csv`


In [ ]:
quality_realized_vol_observed = annualized_realized_vol(
    weekly_log_returns,
    window=QUALITY_WINDOW,
    min_periods=QUALITY_MIN_PERIODS,
)
quality_downside_semivol_observed = rolling_downside_semivol(
    weekly_simple_returns,
    window=QUALITY_WINDOW,
    min_periods=QUALITY_MIN_PERIODS,
)
quality_drawdown_freq_observed = rolling_drawdown_frequency(
    weekly_prices,
    window=QUALITY_DRAWDOWN_WINDOW,
    threshold=-0.05,
    min_periods=max(13, QUALITY_DRAWDOWN_WINDOW // 2),
)
quality_drawdown_severity_observed = rolling_drawdown_severity(
    weekly_prices,
    window=QUALITY_DRAWDOWN_WINDOW,
    min_periods=max(13, QUALITY_DRAWDOWN_WINDOW // 2),
)

quality_inv_vol_score_observed = panel_score(-quality_realized_vol_observed, winsorize=True)
quality_inv_downside_score_observed = panel_score(-quality_downside_semivol_observed, winsorize=True)
quality_inv_drawdown_freq_observed = panel_score(-quality_drawdown_freq_observed, winsorize=True)
quality_inv_drawdown_severity_observed = panel_score(-quality_drawdown_severity_observed, winsorize=True)
quality_score_observed = (
    quality_inv_vol_score_observed
    + quality_inv_downside_score_observed
    + quality_inv_drawdown_freq_observed
    + quality_inv_drawdown_severity_observed
) / 4.0
quality_score_tradable = apply_signal_lag(quality_score_observed)
quality_asset_neutral_observed = grouped_panel_score(
    quality_score_observed,
    asset_class_map,
    min_group_size=ASSET_CLASS_NEUTRAL_MIN_GROUP_SIZE,
)
quality_asset_neutral_tradable = apply_signal_lag(quality_asset_neutral_observed)

quality_long = panel_dict_to_long(
    {
        "quality_realized_vol_ann_observed": quality_realized_vol_observed,
        "quality_downside_semivol_ann_observed": quality_downside_semivol_observed,
        "quality_drawdown_frequency_observed": quality_drawdown_freq_observed,
        "quality_drawdown_severity_observed": quality_drawdown_severity_observed,
        "quality_score_observed": quality_score_observed,
        "quality_score_tradable": quality_score_tradable,
        "quality_score_asset_class_neutral_observed": quality_asset_neutral_observed,
        "quality_score_asset_class_neutral_tradable": quality_asset_neutral_tradable,
    }
).merge(asset_class_frame, on="Ticker", how="left")
quality_path = OUTPUT_DIR / "signal_quality.csv"
quality_long.to_csv(quality_path, index=False)

quality_ic_summary, _, quality_cost_table, _ = register_cross_signal("quality_proxy", quality_score_tradable)

add_manifest_record(
    signal_name="quality_proxy",
    file_name="signal_quality.csv",
    description="ETF quality proxy built from inverse vol, downside semivol, drawdown frequency, and drawdown severity.",
    category="quality_proxy",
    cross_sectional_or_time_series="cross_sectional",
    required_inputs=["weekly_prices.csv", "weekly_returns.csv"],
    lookback="26 to 52 weeks depending on component",
    skip_period="0 weeks",
    lag_applied=total_lag_weeks("price"),
    normalization_method="average of four winsorized cross-sectional stability scores",
    caveats="Captures return-path stability, not classic accounting quality.",
)

if quality_asset_neutral_tradable.notna().sum().sum() > 0:
    quality_asset_ic_summary, _, quality_asset_cost_table, _ = register_cross_signal(
        "quality_proxy_asset_class_neutral",
        quality_asset_neutral_tradable,
    )
    add_manifest_record(
        signal_name="quality_proxy_asset_class_neutral",
        file_name="signal_quality.csv",
        description="ETF quality proxy ranked within asset class using the composite return-path stability score.",
        category="quality_proxy",
        cross_sectional_or_time_series="cross_sectional",
        required_inputs=["weekly_prices.csv", "weekly_returns.csv", "universe_metadata.csv"],
        lookback="26 to 52 weeks depending on component",
        skip_period="0 weeks",
        lag_applied=total_lag_weeks("price"),
        normalization_method="within-asset-class rank to [-1, +1]",
        caveats="Useful when global quality is dominated by structural differences across asset classes rather than within-group stability.",
    )
else:
    quality_asset_ic_summary = pd.DataFrame()
    quality_asset_cost_table = pd.DataFrame()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_signal_examples(quality_score_observed, "Observed quality proxy scores", ax=axes[0])
plot_decay_curve(quality_ic_summary, "mean_ic", "Quality proxy IC decay", ax=axes[1])
plot_decay_curve(quality_asset_ic_summary, "mean_ic", "Quality proxy asset-neutral IC decay", ax=axes[2])
plt.tight_layout()
plt.show()

print(f"Saved: {quality_path}")
print()
print("Global quality proxy")
print(quality_ic_summary.round(4).to_string(index=False))
print(quality_cost_table.round(4).to_string(index=False))
print()
if not quality_asset_ic_summary.empty:
    print("Asset-class-neutral quality proxy")
    print(quality_asset_ic_summary.round(4).to_string(index=False))
    print(quality_asset_cost_table.round(4).to_string(index=False))
else:
    print("Asset-class-neutral quality proxy was skipped because no asset class had enough members.")


## 11. Macro / Alternative Conditioning Signals

These are **market-level regime features**, not standalone cross-sectional ETF alphas.

### VIX term structure / volatility carry

- primary input: `vix_term_structure.csv`
- signal family: contango vs backwardation, slope level, slope z-score
- treatment: **price-based market feature**, so it uses the common 1-week signal lag

### Macro regime score

We combine available risk-off features from:

- VIX level
- VIX slope
- yield-curve slope
- credit spreads
- NFCI if available
- policy-minus-3M short-rate spread if available
- PMI
- dollar strength

Each feature is standardized and signed so that **higher = more risk-off**.

### Google Trends fear / sentiment

If `google_trends.csv` already contains a `fear_composite`, we use it.
Google Trends is treated as an external feature, so it receives the stricter lag policy before downstream use.

**ETF caveats**

These series should usually condition position sizing, signal blending, or risk budgets. They should not be treated as if they directly predict every ETF cross-section every week.

**Output file**

- `data/02_layer1_signals/regime_features.csv`


In [ ]:
regime_features = pd.DataFrame(index=weekly_prices.index)
regime_features.index.name = "Date"

price_feature_signal_cols = []
external_feature_signal_cols = []

if not vix_term.empty:
    if "VIX" in vix_term.columns:
        regime_features["vix_level_observed"] = vix_term["VIX"]
        regime_features["vix_level_z_observed"] = rolling_zscore(vix_term["VIX"])
        regime_features["vix_level_z_tradable"] = apply_signal_lag(regime_features["vix_level_z_observed"])
        price_feature_signal_cols.append("vix_level_z_tradable")
    if "slope_1m_3m" in vix_term.columns:
        regime_features["vix_slope_1m_3m_observed"] = vix_term["slope_1m_3m"]
        regime_features["vix_slope_risk_off_z_observed"] = -rolling_zscore(vix_term["slope_1m_3m"])
        regime_features["vix_slope_risk_off_z_tradable"] = apply_signal_lag(regime_features["vix_slope_risk_off_z_observed"])
        regime_features["vix_contango_flag_observed"] = (vix_term["slope_1m_3m"] > 0).astype(float)
        regime_features["vix_contango_flag_tradable"] = apply_signal_lag(regime_features["vix_contango_flag_observed"])
        price_feature_signal_cols.append("vix_slope_risk_off_z_tradable")

macro_specs = [
    ("T10Y2Y", "yield_curve_risk_off_z_observed", -1.0),
    ("BAMLH0A0HYM2", "credit_spread_risk_off_z_observed", 1.0),
    ("NFCI", "nfci_risk_off_z_observed", 1.0),
    ("policy_minus_3m", "policy_minus_3m_risk_off_z_observed", 1.0),
    ("NAPM", "pmi_risk_off_z_observed", -1.0),
    ("DTWEXBGS", "dollar_risk_off_z_observed", 1.0),
]

for raw_col, out_col, sign in macro_specs:
    if raw_col in macro_weekly.columns:
        regime_features[out_col] = sign * rolling_zscore(macro_weekly[raw_col])
        tradable_col = out_col.replace("_observed", "_tradable")
        regime_features[tradable_col] = apply_signal_lag(
            regime_features[out_col],
            extra_lag_weeks=EXTERNAL_DATA_LAG_WEEKS,
        )
        external_feature_signal_cols.append(tradable_col)

google_fear_source = None
if "fear_composite_zscore" in google_trends.columns:
    regime_features["google_fear_z_observed"] = google_trends["fear_composite_zscore"]
    regime_features["google_fear_z_tradable"] = apply_signal_lag(
        regime_features["google_fear_z_observed"],
        extra_lag_weeks=EXTERNAL_DATA_LAG_WEEKS,
    )
    google_fear_source = "fear_composite_zscore"
    external_feature_signal_cols.append("google_fear_z_tradable")
elif "fear_composite" in google_trends.columns:
    regime_features["google_fear_z_observed"] = rolling_zscore(google_trends["fear_composite"])
    regime_features["google_fear_z_tradable"] = apply_signal_lag(
        regime_features["google_fear_z_observed"],
        extra_lag_weeks=EXTERNAL_DATA_LAG_WEEKS,
    )
    google_fear_source = "fear_composite"
    external_feature_signal_cols.append("google_fear_z_tradable")

composite_cols = price_feature_signal_cols + external_feature_signal_cols
if composite_cols:
    regime_features["macro_risk_score_tradable"] = regime_features[composite_cols].mean(axis=1)
else:
    regime_features["macro_risk_score_tradable"] = np.nan

regime_features["macro_regime_label_tradable"] = pd.cut(
    regime_features["macro_risk_score_tradable"],
    bins=[-np.inf, -0.5, 0.5, np.inf],
    labels=["risk_on", "neutral", "risk_off"],
).astype("object")

regime_path = OUTPUT_DIR / "regime_features.csv"
regime_features.to_csv(regime_path)

add_manifest_record(
    signal_name="vix_term_structure_regime",
    file_name="regime_features.csv",
    description="VIX level and term-structure slope regime features.",
    category="conditioning_feature",
    cross_sectional_or_time_series="conditioning_feature",
    required_inputs=["vix_term_structure.csv"],
    lookback="104-week rolling z-score context",
    skip_period="n/a",
    lag_applied=total_lag_weeks("price"),
    normalization_method="rolling z-scores and flags",
    caveats="Market-level conditioning only; not a per-ETF alpha.",
)
add_manifest_record(
    signal_name="macro_risk_score",
    file_name="regime_features.csv",
    description="Composite macro risk-off score mixing price and lagged external regime features.",
    category="conditioning_feature",
    cross_sectional_or_time_series="conditioning_feature",
    required_inputs=["macro_weekly.csv", "vix_term_structure.csv", "google_trends.csv"],
    lookback="104-week rolling z-score context",
    skip_period="n/a",
    lag_applied=total_lag_weeks("external"),
    normalization_method="average of lagged risk-off z-scores",
    caveats="Conservative timing because macro and Google Trends publication timing is imperfect.",
)
add_manifest_record(
    signal_name="google_fear_regime",
    file_name="regime_features.csv",
    description="Lagged Google Trends fear / sentiment feature when upstream support exists.",
    category="conditioning_feature",
    cross_sectional_or_time_series="conditioning_feature",
    required_inputs=["google_trends.csv"],
    lookback="104-week rolling z-score context",
    skip_period="n/a",
    lag_applied=total_lag_weeks("external"),
    normalization_method="rolling z-score of fear composite",
    caveats="Noisy and potentially unstable; best used as a conditioning feature, not a headline alpha.",
)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
if "macro_risk_score_tradable" in regime_features.columns:
    regime_features["macro_risk_score_tradable"].plot(ax=axes[0], color="steelblue", lw=1.5)
    axes[0].axhline(0.0, color="black", lw=1, alpha=0.5)
    axes[0].set_title("Tradable composite macro risk score")
if "vix_slope_1m_3m_observed" in regime_features.columns:
    regime_features["vix_slope_1m_3m_observed"].plot(ax=axes[1], color="darkred", lw=1.5)
    axes[1].axhline(0.0, color="black", lw=1, alpha=0.5)
    axes[1].set_title("Observed VIX term-structure slope (3M - spot)")
plt.tight_layout()
plt.show()

print(f"Saved: {regime_path}")
print()
print("Google Trends source used:", google_fear_source)
print("Price-based regime signal columns:", price_feature_signal_cols)
print("External regime signal columns:", external_feature_signal_cols)
print()
print(regime_features.tail().to_string())


## 12. Signals Not Implemented Yet Due to Data Constraints

The following signals remain intentionally **not** implemented because the required upstream data are still missing.
This section is a feature, not a bug: it stops the research stack from pretending weak proxies are the same thing as real data.

- **Analyst revisions**
  Need historical consensus estimate changes or recommendation changes at the constituent level, plus a way to aggregate them into ETF exposure.
- **Earnings surprise**
  Need constituent-level earnings calendars, actuals, estimates, and a reproducible aggregation method from issuers to ETF baskets.
- **Accruals**
  Need accounting statement history and stable constituent / holdings history through time.
- **NLP news sentiment**
  Need a timestamped news corpus, entity mapping, and a reproducible NLP pipeline rather than ad hoc live scraping.
- **Satellite / credit-card / job-posting data**
  Need licensed vendor feeds and versioned raw storage in the data hub.

The right path later is to extend the data hub only when the raw inputs can be stored, versioned, and reused cleanly across notebooks.


## 13. Signal Redundancy, Eligibility, and Compact Recommendation

The summary block below does three things that matter for downstream work:

- measures **validation quality**
- measures **redundancy vs distinctiveness**
- records a simple **eligibility matrix** by asset class and data reliability

In ETF universes, the main momentum family often clusters together:

- TSMOM
- XSMOM
- multi-horizon momentum
- residual momentum

That is not automatically bad, but it does mean later composites should avoid pretending four similar trend signals are four independent sources of information.

The pair tables and heatmap below are the actual test to trust. Priors are useful, but the standardized tradable panels are what later notebooks should use when deciding which signals are complementary and which are mostly different wrappers around the same effect.


In [ ]:
cross_summary_table = pd.concat(cross_sectional_summaries, ignore_index=True) if cross_sectional_summaries else pd.DataFrame()
cross_ic_ts_table = pd.concat(cross_sectional_ic_timeseries, ignore_index=True) if cross_sectional_ic_timeseries else pd.DataFrame()
time_summary_table = pd.concat(time_series_summaries, ignore_index=True) if time_series_summaries else pd.DataFrame()
cost_summary_table = pd.concat(cost_summaries, ignore_index=True) if cost_summaries else pd.DataFrame()

ic_summary_path = OUTPUT_DIR / "signal_ic_by_horizon.csv"
cross_summary_table.to_csv(ic_summary_path, index=False)


stacked_signal_panels = {
    name: panel.stack().rename("value").to_frame()
    for name, panel in cross_signal_panels.items()
}


def average_cross_sectional_correlation(stacked_a, stacked_b, min_assets=MIN_CROSS_SECTION):
    joint = stacked_a.join(stacked_b, how="inner", lsuffix="_a", rsuffix="_b").dropna()
    if joint.empty:
        return np.nan
    valid_counts = joint.groupby(level=0).size()
    valid_dates = valid_counts[valid_counts >= min_assets].index
    if len(valid_dates) == 0:
        return np.nan
    joint = joint.loc[joint.index.get_level_values(0).isin(valid_dates)]
    # Small performance cleanup: compute per-date Spearman correlations in grouped form instead of looping date by date for every signal pair.
    grouped_corr = joint.groupby(level=0)[["value_a", "value_b"]].corr(method="spearman")
    pair_corrs = grouped_corr.loc[(slice(None), "value_a"), "value_b"]
    pair_corrs.index = pair_corrs.index.droplevel(1)
    if pair_corrs.empty:
        return np.nan
    return float(pair_corrs.mean())


signal_names_for_redundancy = sorted(cross_signal_panels.keys())
redundancy_matrix = pd.DataFrame(index=signal_names_for_redundancy, columns=signal_names_for_redundancy, dtype=float)
for signal_name in signal_names_for_redundancy:
    redundancy_matrix.loc[signal_name, signal_name] = 1.0
for left_name, right_name in combinations(signal_names_for_redundancy, 2):
    corr = average_cross_sectional_correlation(
        stacked_signal_panels[left_name],
        stacked_signal_panels[right_name],
    )
    redundancy_matrix.loc[left_name, right_name] = corr
    redundancy_matrix.loc[right_name, left_name] = corr

redundancy_path = OUTPUT_DIR / "signal_redundancy_matrix.csv"
redundancy_matrix.to_csv(redundancy_path)

pair_rows = []
for left_name, right_name in combinations(signal_names_for_redundancy, 2):
    corr = redundancy_matrix.loc[left_name, right_name]
    pair_rows.append(
        {
            "left_signal": left_name,
            "right_signal": right_name,
            "avg_cs_corr": corr,
            "abs_avg_cs_corr": abs(corr) if pd.notna(corr) else np.nan,
        }
    )
pair_table = pd.DataFrame(pair_rows)
if pair_table.empty:
    pair_table = pd.DataFrame(columns=["left_signal", "right_signal", "avg_cs_corr", "abs_avg_cs_corr"])
else:
    pair_table = pair_table.sort_values("abs_avg_cs_corr", ascending=False)
redundancy_pairs_path = OUTPUT_DIR / "signal_redundancy_pairs.csv"
pair_table.to_csv(redundancy_pairs_path, index=False)

most_redundant_pairs = pair_table.head(5)
most_distinct_pairs = pair_table.sort_values("abs_avg_cs_corr", ascending=True).head(5)

fig, ax = plt.subplots(figsize=(12, 10))
matrix_values = redundancy_matrix.astype(float).values
im = ax.imshow(matrix_values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(redundancy_matrix.columns)))
ax.set_yticks(range(len(redundancy_matrix.index)))
ax.set_xticklabels(redundancy_matrix.columns, rotation=90)
ax.set_yticklabels(redundancy_matrix.index)
ax.set_title("Average cross-sectional signal correlation over time")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

redundancy_notes = []
if not most_redundant_pairs.empty and pd.notna(most_redundant_pairs.iloc[0]["avg_cs_corr"]):
    row = most_redundant_pairs.iloc[0]
    redundancy_notes.append(
        f"Closest tradable pair: {row['left_signal']} vs {row['right_signal']} (avg cross-sectional corr = {row['avg_cs_corr']:.2f})."
    )
if not most_distinct_pairs.empty and pd.notna(most_distinct_pairs.iloc[0]["avg_cs_corr"]):
    row = most_distinct_pairs.iloc[0]
    redundancy_notes.append(
        f"Most distinct tradable pair: {row['left_signal']} vs {row['right_signal']} (avg cross-sectional corr = {row['avg_cs_corr']:.2f})."
    )
momentum_family = [name for name in ["tsmom_vol_scaled", "xsmom_global", "multi_mom_invvol", "residual_momentum"] if name in redundancy_matrix.index]
if len(momentum_family) >= 2:
    momentum_cluster_score = redundancy_matrix.loc[momentum_family, momentum_family].replace(1.0, np.nan).abs().stack().mean()
    redundancy_notes.append(
        f"Momentum-family signals show average absolute cross-correlation of {momentum_cluster_score:.2f}; later composites should treat them as related rather than fully independent."
    )

all_signal_names = sorted(
    set(cross_summary_table.get("signal_name", pd.Series(dtype=str)).dropna().tolist())
    | set(time_summary_table.get("signal_name", pd.Series(dtype=str)).dropna().tolist())
    | set(cost_summary_table.get("signal_name", pd.Series(dtype=str)).dropna().tolist())
)

signal_summary = pd.DataFrame(index=all_signal_names)

if not cross_summary_table.empty:
    cross_agg = cross_summary_table.groupby("signal_name").agg(
        avg_mean_ic=("mean_ic", "mean"),
        avg_ic_tstat=("ic_tstat", "mean"),
        avg_ic_tstat_nw=("ic_tstat_nw", "mean"),
        positive_horizon_share=("mean_ic", lambda s: (s > 0).mean()),
        avg_cross_coverage=("mean_coverage", "mean"),
    )
    signal_summary = signal_summary.join(cross_agg, how="left")

if not time_summary_table.empty:
    time_agg = time_summary_table.groupby("signal_name").agg(
        avg_signed_strategy_mean=("signed_strategy_mean", "mean"),
        avg_signed_strategy_tstat=("signed_strategy_tstat", "mean"),
        avg_signed_strategy_tstat_nw=("signed_strategy_tstat_nw", "mean"),
        avg_directional_hit_rate=("directional_hit_rate", "mean"),
        positive_ts_horizon_share=("signed_strategy_mean", lambda s: (s > 0).mean()),
    )
    signal_summary = signal_summary.join(time_agg, how="left")

if not cost_summary_table.empty:
    cost_10 = cost_summary_table[cost_summary_table["cost_bps"] == 10].set_index("signal_name")
    signal_summary = signal_summary.join(
        cost_10[["ann_return", "ann_vol", "sharpe", "avg_weekly_turnover"]].rename(
            columns={
                "ann_return": "ann_return_10bps",
                "ann_vol": "ann_vol_10bps",
                "sharpe": "net_sharpe_10bps",
            }
        ),
        how="left",
    )

if not redundancy_matrix.empty:
    avg_abs_corr = redundancy_matrix.abs().replace(1.0, np.nan).mean(axis=1)
    signal_summary["avg_abs_redundancy"] = avg_abs_corr
    signal_summary["distinctiveness_score"] = 1.0 - avg_abs_corr

for required_col in [
    "avg_ic_tstat",
    "avg_ic_tstat_nw",
    "avg_signed_strategy_tstat",
    "avg_signed_strategy_tstat_nw",
    "positive_horizon_share",
    "positive_ts_horizon_share",
    "net_sharpe_10bps",
    "avg_weekly_turnover",
    "avg_cross_coverage",
    "distinctiveness_score",
]:
    if required_col not in signal_summary.columns:
        signal_summary[required_col] = np.nan

signal_summary["primary_validation_stat"] = (
    signal_summary["avg_ic_tstat_nw"]
    .fillna(signal_summary["avg_ic_tstat"])
    .fillna(signal_summary["avg_signed_strategy_tstat_nw"])
    .fillna(signal_summary["avg_signed_strategy_tstat"])
)
signal_summary["robust_horizon_share"] = signal_summary["positive_horizon_share"].fillna(signal_summary["positive_ts_horizon_share"])
signal_summary["validation_quality_score"] = (
    signal_summary["primary_validation_stat"].fillna(0.0)
    + 0.5 * signal_summary["robust_horizon_share"].fillna(0.0)
    + 0.5 * signal_summary["net_sharpe_10bps"].fillna(0.0)
    + 0.25 * signal_summary["distinctiveness_score"].fillna(0.0)
    - 0.1 * signal_summary["avg_weekly_turnover"].fillna(0.0)
)
signal_summary = signal_summary.sort_values("validation_quality_score", ascending=False).reset_index().rename(columns={"index": "signal_name"})

def recommendation_label(row):
    if pd.notna(row["validation_quality_score"]) and row["validation_quality_score"] >= signal_summary["validation_quality_score"].quantile(0.75):
        return "strong"
    if pd.notna(row["validation_quality_score"]) and row["validation_quality_score"] <= signal_summary["validation_quality_score"].quantile(0.25):
        return "weak"
    if pd.notna(row["robust_horizon_share"]) and row["robust_horizon_share"] >= 0.8:
        return "robust"
    return "mixed"

signal_summary["recommendation"] = signal_summary.apply(recommendation_label, axis=1)

coverage_lookup = {}
if not cross_summary_table.empty:
    coverage_lookup = (
        cross_summary_table[cross_summary_table["horizon_weeks"] == 1]
        .set_index("signal_name")["mean_coverage"]
        .div(len(research_universe))
        .to_dict()
    )

eligibility_template = [
    {"signal_name": "tsmom_vol_scaled", "equities": "High", "bonds": "High", "reits": "Medium", "commodities": "High", "fx": "Medium", "base_proxy_quality": "High", "notes": "Clean price-based trend signal."},
    {"signal_name": "xsmom_global", "equities": "High", "bonds": "High", "reits": "Low", "commodities": "High", "fx": "Low", "base_proxy_quality": "High", "notes": "Global relative-strength signal across the universe."},
    {"signal_name": "xsmom_asset_class_neutral", "equities": "High", "bonds": "High", "reits": "Low", "commodities": "Medium", "fx": "Low", "base_proxy_quality": "Medium", "notes": "Useful when broad asset-class rotation dominates."},
    {"signal_name": "reversal_1w_global", "equities": "Medium", "bonds": "Medium", "reits": "Low", "commodities": "Medium", "fx": "Low", "base_proxy_quality": "High", "notes": "Fast and regime-sensitive."},
    {"signal_name": "reversal_1w_asset_class_neutral", "equities": "Medium", "bonds": "Medium", "reits": "Low", "commodities": "Low", "fx": "Low", "base_proxy_quality": "Medium", "notes": "Cleaner within-group reversal where enough names exist."},
    {"signal_name": "reversal_4w_global", "equities": "Medium", "bonds": "Medium", "reits": "Low", "commodities": "Medium", "fx": "Low", "base_proxy_quality": "High", "notes": "Slower reversal proxy."},
    {"signal_name": "reversal_4w_asset_class_neutral", "equities": "Medium", "bonds": "Medium", "reits": "Low", "commodities": "Low", "fx": "Low", "base_proxy_quality": "Medium", "notes": "Neutral version for slower reversal."},
    {"signal_name": "multi_mom_equal", "equities": "High", "bonds": "High", "reits": "Medium", "commodities": "High", "fx": "Medium", "base_proxy_quality": "High", "notes": "Blended trend signal."},
    {"signal_name": "multi_mom_invvol", "equities": "High", "bonds": "High", "reits": "Medium", "commodities": "High", "fx": "Medium", "base_proxy_quality": "High", "notes": "More stable blended trend signal."},
    {"signal_name": "residual_momentum", "equities": "High", "bonds": "Medium", "reits": "Medium", "commodities": "Medium", "fx": "Low", "base_proxy_quality": "Medium", "notes": f"Depends on explicit market proxy {market_proxy_ticker}."},
    {"signal_name": "carry_proxy", "equities": "Low", "bonds": "High", "reits": "Medium", "commodities": "Low", "fx": "Low", "base_proxy_quality": "Low", "notes": "Proxy only; strongest where distributions are economically meaningful."},
    {"signal_name": "carry_proxy_asset_class_neutral", "equities": "Low", "bonds": "High", "reits": "Medium", "commodities": "Low", "fx": "Low", "base_proxy_quality": "Low", "notes": "Within-asset-class version of the carry proxy."},
    {"signal_name": "value_proxy", "equities": "Medium", "bonds": "Medium", "reits": "Medium", "commodities": "Low", "fx": "Low", "base_proxy_quality": "Medium", "notes": "Own-history value, not balance-sheet value."},
    {"signal_name": "value_proxy_asset_class_neutral", "equities": "Medium", "bonds": "Medium", "reits": "Medium", "commodities": "Low", "fx": "Low", "base_proxy_quality": "Medium", "notes": "Within-asset-class version of own-history value."},
    {"signal_name": "bab_proxy", "equities": "High", "bonds": "Medium", "reits": "Low", "commodities": "Low", "fx": "Low", "base_proxy_quality": "Medium", "notes": "Low-beta proxy, not full BAB construction."},
    {"signal_name": "bab_proxy_asset_class_neutral", "equities": "High", "bonds": "Medium", "reits": "Low", "commodities": "Low", "fx": "Low", "base_proxy_quality": "Medium", "notes": "Within-asset-class low-beta ranking."},
    {"signal_name": "quality_proxy", "equities": "Medium", "bonds": "Medium", "reits": "Medium", "commodities": "Medium", "fx": "Low", "base_proxy_quality": "High", "notes": "Return-path stability proxy."},
    {"signal_name": "quality_proxy_asset_class_neutral", "equities": "Medium", "bonds": "Medium", "reits": "Medium", "commodities": "Medium", "fx": "Low", "base_proxy_quality": "High", "notes": "Within-asset-class quality ranking."},
    {"signal_name": "vix_term_structure_regime", "equities": "Conditioning", "bonds": "Conditioning", "reits": "Conditioning", "commodities": "Conditioning", "fx": "Conditioning", "base_proxy_quality": "High", "notes": "Market-level conditioning feature, not a per-ETF alpha."},
    {"signal_name": "macro_risk_score", "equities": "Conditioning", "bonds": "Conditioning", "reits": "Conditioning", "commodities": "Conditioning", "fx": "Conditioning", "base_proxy_quality": "Medium", "notes": "Macro conditioning signal with conservative lagging."},
    {"signal_name": "google_fear_regime", "equities": "Conditioning", "bonds": "Conditioning", "reits": "Conditioning", "commodities": "Conditioning", "fx": "Conditioning", "base_proxy_quality": "Low", "notes": "Noisy fear / sentiment conditioning feature."},
]

def reliability_label(base_proxy_quality, coverage_ratio):
    score = {"High": 2, "Medium": 1, "Low": 0}.get(base_proxy_quality, 1)
    if pd.notna(coverage_ratio):
        if coverage_ratio >= 0.8:
            score += 1
        elif coverage_ratio < 0.4:
            score -= 1
    score = max(0, min(3, score))
    return {3: "High", 2: "Medium-High", 1: "Medium", 0: "Low"}[score]

eligibility_rows = []
for row in eligibility_template:
    coverage_ratio = coverage_lookup.get(row["signal_name"], np.nan)
    row["coverage_ratio_h1"] = coverage_ratio
    row["reliability"] = reliability_label(row["base_proxy_quality"], coverage_ratio)
    eligibility_rows.append(row)
eligibility_df = pd.DataFrame(eligibility_rows)
eligibility_path = OUTPUT_DIR / "signal_eligibility_matrix.csv"
eligibility_df.to_csv(eligibility_path, index=False)

summary_path = OUTPUT_DIR / "signal_summary_table.csv"
signal_summary.to_csv(summary_path, index=False)

manifest_path = OUTPUT_DIR / "signal_manifest.json"
manifest_records = []
seen = set()
for record in signal_manifest_records:
    key = record.get("signal_name")
    if key in seen:
        continue
    manifest_records.append(record)
    seen.add(key)
manifest_path.write_text(json.dumps(manifest_records, indent=2))

strongest = signal_summary.head(3)[["signal_name", "validation_quality_score", "primary_validation_stat", "net_sharpe_10bps", "avg_weekly_turnover"]]
weakest = signal_summary.tail(3)[["signal_name", "validation_quality_score", "primary_validation_stat", "net_sharpe_10bps", "avg_weekly_turnover"]]
robust = signal_summary.sort_values(
    ["robust_horizon_share", "distinctiveness_score", "avg_cross_coverage"],
    ascending=[False, False, False],
).head(3)[["signal_name", "robust_horizon_share", "distinctiveness_score", "avg_cross_coverage"]]

print(f"Saved: {ic_summary_path}")
print(f"Saved: {redundancy_path}")
print(f"Saved: {redundancy_pairs_path}")
print(f"Saved: {eligibility_path}")
print(f"Saved: {summary_path}")
print(f"Saved: {manifest_path}")
print()
print("Redundancy comments")
for note in redundancy_notes:
    print(" -", note)
print()
print("Most redundant signal pairs")
print(most_redundant_pairs.round(4).to_string(index=False))
print()
print("Most distinct signal pairs")
print(most_distinct_pairs.round(4).to_string(index=False))
print()
print("Eligibility matrix")
print(eligibility_df.round(4).to_string(index=False))
print()
print("Full ranking")
print(signal_summary.round(4).to_string(index=False))
print()
print("Strongest signals")
print(strongest.round(4).to_string(index=False))
print()
print("Weakest signals")
print(weakest.round(4).to_string(index=False))
print()
print("Most robust / distinct signals")
print(robust.round(4).to_string(index=False))
print()
print("Files now in output directory:")
for file in sorted(OUTPUT_DIR.glob("signal_*.csv")) + sorted(OUTPUT_DIR.glob("signal_*.json")) + sorted(OUTPUT_DIR.glob("regime_features.csv")):
    print("  -", file.name)


## 14. Data Hub Audit / Suggested Improvements

Reviewing `01_data_hub.ipynb` as the upstream design reference surfaced a few improvements that materially help downstream signal research.
The useful question now is not “what should exist in theory?”, but “what was implemented cleanly and what still needs a stronger upstream data source?”

### Implemented upstream now

| Item | Status | Why it matters downstream |
|---|---|---|
| `daily_returns.csv` | Implemented | BAB, vol, downside, and later risk work no longer need to recompute daily returns from scratch. |
| Explicit benchmark files plus `proxy_mapping.json` | Implemented | Later notebooks do not need to assume `SPY` implicitly for market beta or cash / duration references. |
| `universe_metadata.csv` with asset-class labels | Implemented | Asset-class-neutral momentum and reversal now have a clean upstream label source. |
| Cached Yahoo metadata snapshot | Implemented | Carry and audit work can prefer a stored snapshot over live metadata lookups. |
| Cached ETF distribution history snapshot | Implemented | Carry research now has a reproducible historical proxy input rather than relying only on live point-in-time metadata. |
| Raw Google Trends snapshot plus pull metadata | Implemented | Fear / sentiment features are easier to audit and less dependent on repeated live pulls. |
| Added reproducible liquidity / short-rate proxies (`NFCI`, `DGS3MO`, `policy_minus_3m`) | Implemented | Macro regime conditioning is stronger without needing brittle one-off data sources. |
| Stable numbered folder convention under `data/01_data_hub/` and `data/02_layer1_signals/` | Implemented | Makes notebook chaining and artifact discovery much cleaner. |

### Still recommended, but not implemented yet

| Item | Why it still matters | Why it was not implemented now |
|---|---|---|
| A clean MOVE source or other bond-volatility proxy | Would improve duration / rates regime conditioning materially | Not wired because the current stack does not yet have a clearly reproducible, versioned free source for MOVE. |
| Constituent holdings history and ETF fundamentals | Needed for stronger value, carry, earnings, accrual, and revision-style signals | The current hub is ETF-price-first and does not yet store constituent histories through time. |
| Release-timestamp-aware macro lagging | Would tighten macro timing even further | The notebook now uses a conservative one-week extra lag, which is a strong improvement, but not a full release-calendar model. |
| Richer commodity / FX carry inputs | Would make carry research more honest for commodities and FX | ETF distributions are not the same as true futures roll or currency carry. Better raw data are still required. |

**Bottom line**

The data hub is materially better connected to Layer 1 now.
The biggest remaining limitation is not code structure — it is the availability of richer point-in-time data for truly fundamental, constituent-level, or true-carry signals.


## 15. Final Summary

1. **What was improved in this final Layer 1 polish pass**
   The notebook now has an even clearer global timing section near the top, fully validated residual momentum, asset-class-neutral variants for carry / value / BAB / quality where the labels support them, stronger redundancy outputs for later signal-combination work, and a cleaner machine-readable manifest for downstream notebooks.

2. **Why those improvements matter**
   They make Layer 1 easier to trust, easier for Layer 2 to consume, and less dependent on hidden notebook assumptions. The notebook now exposes both global and within-asset-class cross-sectional views without replacing one with the other.

3. **Which improvements most reduce the risk of false confidence in backtests**
   The most important protections are still the explicit `SIGNAL_LAG_WEEKS` rule, the extra lag on non-price external features, validating tradable rather than same-date observed signals, and treating high within-family signal correlation as a warning against counting related momentum signals as independent evidence.

4. **Which signals now look most promising**
   The most promising families are still the cleaner price-based signals: multi-horizon momentum, residual momentum, XSMOM, and the better-behaved quality / low-beta style proxies. The actual summary table should drive the final ranking, but those are the sleeves most likely to remain useful in later Layer 2 combinations.

5. **Which limitations are still mainly data-related**
   Carry remains the most data-limited implemented signal because ETF distribution yield is only a proxy for true carry, especially in commodities and FX. The value proxy is still “own-history value,” not constituent fundamentals. Google Trends is intentionally noisy and best treated as conditioning. The unimplemented analyst / earnings / accrual / NLP / alternative-data signals still need genuinely richer upstream data before they should be built.
